In [1]:
# ================================================================
# 19 - PRODUCTION ML MODEL PIPELINE
# AI-Based Instagram Engagement Prediction and Content Optimization
# ================================================================

from pathlib import Path
import json
import warnings
import joblib
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

print("=" * 70)
print("PRODUCTION ML MODEL PIPELINE")
print("=" * 70)


# ================================================================
# 1. PROJECT ROOT
# ================================================================

PROJECT_ROOT = Path(
    r"d:\newwwwwwwww\AiBasedInstagramPrediction"
)

print("\nProject root:")
print(PROJECT_ROOT)

if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        f"Project root does not exist:\n{PROJECT_ROOT}"
    )


# ================================================================
# 2. SEARCH FOR SAVED MODELS
# ================================================================

print("\n" + "=" * 70)
print("SEARCHING FOR SAVED MODELS")
print("=" * 70)

all_model_files = []

for extension in [
    "*.joblib",
    "*.pkl",
    "*.pickle"
]:

    all_model_files.extend(
        PROJECT_ROOT.rglob(extension)
    )

all_model_files = sorted(
    set(all_model_files),
    key=lambda x: str(x).lower()
)

print(
    f"\nModel files found: "
    f"{len(all_model_files)}"
)

for i, path in enumerate(
    all_model_files,
    1
):

    print(
        f"{i:02d}. {path}"
    )


# ================================================================
# 3. SEARCH FOR MODEL-RELATED JSON FILES
# ================================================================

print("\n" + "=" * 70)
print("SEARCHING FOR MODEL METADATA")
print("=" * 70)

json_files = sorted(
    [
        p
        for p in PROJECT_ROOT.rglob("*.json")
        if (
            "model" in p.name.lower()
            or "feature" in p.name.lower()
            or "preprocess" in p.name.lower()
        )
    ],
    key=lambda x: str(x).lower()
)

print(
    f"\nModel-related JSON files: "
    f"{len(json_files)}"
)

for path in json_files:

    print(path)


# ================================================================
# 4. IDENTIFY LIKELY FINAL MODEL
# ================================================================

print("\n" + "=" * 70)
print("IDENTIFYING FINAL MODEL")
print("=" * 70)

model_candidates = []

for path in all_model_files:

    name = path.name.lower()

    score = 0

    # Strong indicators
    if "final" in name:
        score += 10

    if "best" in name:
        score += 9

    if "production" in name:
        score += 10

    if "champion" in name:
        score += 10

    if "histgradient" in name:
        score += 8

    if "gradient" in name:
        score += 5

    if "model" in name:
        score += 3

    if "synthetic" in name:
        score += 3

    model_candidates.append(
        (score, path)
    )


model_candidates.sort(
    key=lambda x: (
        -x[0],
        str(x[1]).lower()
    )
)


print(
    "\nRanked model candidates:"
)

for rank, (score, path) in enumerate(
    model_candidates,
    1
):

    print(
        f"{rank:02d}. "
        f"Score={score:02d} | "
        f"{path}"
    )


if not model_candidates:

    raise FileNotFoundError(
        """
No saved ML model was found.

We need to locate the notebook/output where the
90%+ HistGradientBoosting model was saved.
"""
    )


# ================================================================
# 5. LOAD TOP CANDIDATE
# ================================================================

MODEL_FILE = model_candidates[0][1]

print("\n" + "=" * 70)
print("SELECTED MODEL")
print("=" * 70)

print(
    "\nSelected:"
)

print(
    MODEL_FILE
)


try:

    model_object = joblib.load(
        MODEL_FILE
    )

except Exception as e:

    print(
        "\nCould not load using joblib."
    )

    print(
        f"Error: {e}"
    )

    raise


# ================================================================
# 6. INSPECT MODEL OBJECT
# ================================================================

print("\n" + "=" * 70)
print("MODEL OBJECT INSPECTION")
print("=" * 70)

print(
    "\nPython type:"
)

print(
    type(model_object)
)


print(
    "\nModel representation:"
)

print(
    model_object
)


# ================================================================
# 7. CHECK PIPELINE
# ================================================================

print("\n" + "=" * 70)
print("PIPELINE INSPECTION")
print("=" * 70)

if hasattr(
    model_object,
    "steps"
):

    print(
        "\n✓ Saved object is a Pipeline."
    )

    print(
        "\nPipeline steps:"
    )

    for name, component in (
        model_object.steps
    ):

        print(
            f"  {name:<25} "
            f"{type(component).__name__}"
        )

else:

    print(
        "\nSaved object is not a sklearn Pipeline."
    )


# ================================================================
# 8. CHECK MODEL TYPE
# ================================================================

print("\n" + "=" * 70)
print("MODEL TYPE")
print("=" * 70)

actual_model = model_object

if hasattr(
    model_object,
    "steps"
):

    actual_model = (
        model_object.steps[-1][1]
    )

print(
    "\nFinal estimator:"
)

print(
    type(actual_model)
)


# ================================================================
# 9. CHECK HISTGRADIENTBOOSTING
# ================================================================

model_name = (
    type(actual_model).__name__
)

print(
    "\nEstimator name:"
)

print(
    model_name
)


if (
    "HistGradientBoosting"
    in model_name
):

    print(
        "\n✓ HistGradientBoosting confirmed."
    )

else:

    print(
        "\n⚠ Selected model is not "
        "HistGradientBoosting."
    )


# ================================================================
# 10. MODEL PARAMETERS
# ================================================================

print("\n" + "=" * 70)
print("MODEL PARAMETERS")
print("=" * 70)

if hasattr(
    actual_model,
    "get_params"
):

    params = actual_model.get_params()

    for key, value in params.items():

        print(
            f"{key}: {value}"
        )


# ================================================================
# 11. CHECK CLASSES
# ================================================================

print("\n" + "=" * 70)
print("MODEL CLASSES")
print("=" * 70)

if hasattr(
    actual_model,
    "classes_"
):

    print(
        "\nClasses:"
    )

    print(
        actual_model.classes_
    )

else:

    print(
        "\nClasses are not directly exposed."
    )


# ================================================================
# 12. CHECK FEATURE INFORMATION
# ================================================================

print("\n" + "=" * 70)
print("FEATURE INFORMATION")
print("=" * 70)

feature_count = None

if hasattr(
    actual_model,
    "n_features_in_"
):

    feature_count = (
        actual_model.n_features_in_
    )

    print(
        f"\nModel input features: "
        f"{feature_count}"
    )

elif hasattr(
    model_object,
    "n_features_in_"
):

    feature_count = (
        model_object.n_features_in_
    )

    print(
        f"\nPipeline input features: "
        f"{feature_count}"
    )

else:

    print(
        "\nFeature count unavailable."
    )


# ================================================================
# 13. SEARCH FEATURE METADATA
# ================================================================

print("\n" + "=" * 70)
print("SEARCHING FEATURE DEFINITIONS")
print("=" * 70)

feature_files = []

for path in PROJECT_ROOT.rglob("*"):

    if not path.is_file():
        continue

    name = path.name.lower()

    if (
        "feature" in name
        or "column" in name
        or "preprocess" in name
    ):

        if path.suffix.lower() in [
            ".json",
            ".csv",
            ".txt",
            ".joblib",
            ".pkl"
        ]:

            feature_files.append(path)


for path in sorted(
    set(feature_files),
    key=lambda x: str(x).lower()
):

    print(path)


# ================================================================
# 14. LOAD POSSIBLE FEATURE JSON
# ================================================================

feature_metadata = None
feature_metadata_file = None

for path in json_files:

    try:

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)

        text = json.dumps(
            data
        ).lower()

        if (
            "feature" in text
            or "column" in text
        ):

            feature_metadata = data
            feature_metadata_file = path

            break

    except Exception:

        continue


if feature_metadata is not None:

    print("\n" + "=" * 70)
    print("FEATURE METADATA FOUND")
    print("=" * 70)

    print(
        f"\nFile:"
    )

    print(
        feature_metadata_file
    )

    print(
        "\nMetadata:"
    )

    print(
        json.dumps(
            feature_metadata,
            indent=4
        )[:10000]
    )

else:

    print(
        "\nNo suitable feature metadata JSON "
        "was automatically identified."
    )


# ================================================================
# 15. SAVE PRODUCTION MODEL COPY
# ================================================================

PRODUCTION_DIR = (
    PROJECT_ROOT
    / "models"
    / "production"
)

PRODUCTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PRODUCTION_MODEL_FILE = (
    PRODUCTION_DIR
    / "instagram_engagement_model.joblib"
)

joblib.dump(
    model_object,
    PRODUCTION_MODEL_FILE
)

print("\n" + "=" * 70)
print("PRODUCTION MODEL SAVED")
print("=" * 70)

print(
    "\nProduction model:"
)

print(
    PRODUCTION_MODEL_FILE
)


# ================================================================
# 16. CREATE MODEL METADATA
# ================================================================

production_metadata = {

    "project":
        "AI-Based Instagram Engagement Prediction and Content Optimization System",

    "source_model":
        str(MODEL_FILE),

    "production_model":
        str(PRODUCTION_MODEL_FILE),

    "model_type":
        type(actual_model).__name__,

    "feature_count":
        int(feature_count)
        if feature_count is not None
        else None,

    "classes":
        (
            actual_model.classes_.tolist()
            if hasattr(
                actual_model,
                "classes_"
            )
            else None
        ),

    "development_dataset":
        "Synthetic Instagram Engagement Dataset V2",

    "production_role":
        "Primary engagement prediction model",

    "target":
        "performance_class",

    "target_classes":
        [
            "Low",
            "Medium",
            "High"
        ],

    "known_validation_accuracy":
        0.90965,

    "known_validation_weighted_f1":
        0.909996,

    "note":
        "Model selected from completed synthetic "
        "model development and validation."
}


PRODUCTION_METADATA_FILE = (
    PRODUCTION_DIR
    / "instagram_engagement_model_metadata.json"
)

with open(
    PRODUCTION_METADATA_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        production_metadata,
        f,
        indent=4
    )


# ================================================================
# 17. FINAL VALIDATION
# ================================================================

print("\n" + "=" * 70)
print("PRODUCTION MODEL VALIDATION")
print("=" * 70)

print(
    "\nModel file exists:"
)

print(
    PRODUCTION_MODEL_FILE.exists()
)

print(
    "\nModel size:"
)

print(
    f"{PRODUCTION_MODEL_FILE.stat().st_size:,} bytes"
)

print(
    "\nMetadata file exists:"
)

print(
    PRODUCTION_METADATA_FILE.exists()
)


# ================================================================
# 18. RELOAD TEST
# ================================================================

print("\n" + "=" * 70)
print("MODEL RELOAD TEST")
print("=" * 70)

reloaded_model = joblib.load(
    PRODUCTION_MODEL_FILE
)

print(
    "\n✓ Production model successfully reloaded."
)

print(
    "\nReloaded model type:"
)

print(
    type(reloaded_model)
)


# ================================================================
# 19. FINAL STATUS
# ================================================================

print("\n" + "=" * 70)
print("STAGE 1 — PRODUCTION ML PIPELINE")
print("=" * 70)

print(
    """
STATUS

✓ Existing model searched
✓ Best available model identified
✓ Model loaded
✓ Model inspected
✓ Production copy created
✓ Metadata created
✓ Reload test passed

The trained synthetic model is now prepared
for system integration.
"""
)

print("=" * 70)

print(
    "\nNEXT STAGE:"
)

print(
    "STAGE 2 — IMAGE INTEGRATION"
)

print("=" * 70)

PRODUCTION ML MODEL PIPELINE

Project root:
d:\newwwwwwwww\AiBasedInstagramPrediction

SEARCHING FOR SAVED MODELS

Model files found: 0

SEARCHING FOR MODEL METADATA

Model-related JSON files: 0

IDENTIFYING FINAL MODEL

Ranked model candidates:


FileNotFoundError: 
No saved ML model was found.

We need to locate the notebook/output where the
90%+ HistGradientBoosting model was saved.


In [2]:
from pathlib import Path

PROJECT_ROOT = Path(
    r"d:\newwwwwwwww\AiBasedInstagramPrediction"
)

print("=" * 70)
print("FULL PROJECT FILE DISCOVERY")
print("=" * 70)

# ------------------------------------------------
# 1. Search the project and nearby parent folders
# ------------------------------------------------

SEARCH_ROOTS = [
    PROJECT_ROOT,
    PROJECT_ROOT.parent,
]

patterns = [
    "*.joblib",
    "*.pkl",
    "*.pickle",
    "*.json",
]

found = set()

for root in SEARCH_ROOTS:

    if not root.exists():
        continue

    print(f"\nSearching: {root}")

    for pattern in patterns:

        for path in root.rglob(pattern):

            found.add(path)

# ------------------------------------------------
# 2. Display model files
# ------------------------------------------------

print("\n" + "=" * 70)
print("MODEL FILES")
print("=" * 70)

model_files = [
    p for p in found
    if p.suffix.lower() in {
        ".joblib",
        ".pkl",
        ".pickle"
    }
]

if model_files:

    for p in sorted(
        model_files,
        key=lambda x: str(x).lower()
    ):
        print(p)

else:

    print("NO MODEL FILES FOUND")

# ------------------------------------------------
# 3. Display JSON files
# ------------------------------------------------

print("\n" + "=" * 70)
print("JSON FILES")
print("=" * 70)

json_files = [
    p for p in found
    if p.suffix.lower() == ".json"
]

if json_files:

    for p in sorted(
        json_files,
        key=lambda x: str(x).lower()
    ):
        print(p)

else:

    print("NO JSON FILES FOUND")

# ------------------------------------------------
# 4. Search ALL project files for model-related names
# ------------------------------------------------

print("\n" + "=" * 70)
print("MODEL-RELATED FILES")
print("=" * 70)

keywords = [
    "model",
    "gradient",
    "hist",
    "champion",
    "final",
    "feature"
]

related = set()

for root in SEARCH_ROOTS:

    if not root.exists():
        continue

    for path in root.rglob("*"):

        if not path.is_file():
            continue

        name = path.name.lower()

        if any(
            keyword in name
            for keyword in keywords
        ):

            related.add(path)

if related:

    for p in sorted(
        related,
        key=lambda x: str(x).lower()
    ):

        print(p)

else:

    print("NO MODEL-RELATED FILES FOUND")

# ------------------------------------------------
# 5. Search notebooks
# ------------------------------------------------

print("\n" + "=" * 70)
print("NOTEBOOKS")
print("=" * 70)

notebooks = []

for root in SEARCH_ROOTS:

    if not root.exists():
        continue

    notebooks.extend(
        root.rglob("*.ipynb")
    )

for p in sorted(
    set(notebooks),
    key=lambda x: str(x).lower()
):

    print(p)

print("\n" + "=" * 70)
print("DISCOVERY COMPLETE")
print("=" * 70)

FULL PROJECT FILE DISCOVERY

Searching: d:\newwwwwwwww\AiBasedInstagramPrediction

Searching: d:\newwwwwwwww

MODEL FILES
NO MODEL FILES FOUND

JSON FILES
NO JSON FILES FOUND

MODEL-RELATED FILES
NO MODEL-RELATED FILES FOUND

NOTEBOOKS

DISCOVERY COMPLETE


In [3]:
# ================================================================
# PRODUCTION MODEL REBUILD - DATASET INSPECTION
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 70)
print("PRODUCTION MODEL REBUILD")
print("=" * 70)

PROJECT_ROOT = Path(
    r"d:\newwwwwwwww\AiBasedInstagramPrediction"
)

# ------------------------------------------------
# FIND SYNTHETIC V2 DATASET
# ------------------------------------------------

print("\n" + "=" * 70)
print("SEARCHING FOR SYNTHETIC V2 DATASET")
print("=" * 70)

dataset_candidates = []

for p in PROJECT_ROOT.rglob("*.csv"):

    name = p.name.lower()

    if (
        "synthetic" in name
        or "engagement" in name
        or "v2" in name
    ):

        dataset_candidates.append(p)

for i, p in enumerate(
    sorted(
        dataset_candidates,
        key=lambda x: str(x).lower()
    ),
    1
):

    print(
        f"{i:02d}. {p}"
    )

if not dataset_candidates:

    raise FileNotFoundError(
        "Synthetic dataset could not be found."
    )

# Prefer the exact V2 dataset if available

v2_candidates = [
    p for p in dataset_candidates
    if "v2" in p.name.lower()
]

if v2_candidates:

    DATASET_FILE = sorted(
        v2_candidates,
        key=lambda x: str(x).lower()
    )[0]

else:

    DATASET_FILE = sorted(
        dataset_candidates,
        key=lambda x: str(x).lower()
    )[0]

print("\nSelected dataset:")
print(DATASET_FILE)

# ------------------------------------------------
# LOAD
# ------------------------------------------------

df = pd.read_csv(
    DATASET_FILE
)

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print(
    f"\nRows    : {df.shape[0]:,}"
)

print(
    f"Columns : {df.shape[1]}"
)

print(
    "\nTarget candidates:"
)

for col in df.columns:

    if any(
        word in col.lower()
        for word in [
            "target",
            "performance",
            "engagement",
            "class"
        ]
    ):

        print(
            f"  - {col}"
        )

# ------------------------------------------------
# TARGET
# ------------------------------------------------

TARGET = "performance_class"

if TARGET not in df.columns:

    raise ValueError(
        f"Target column '{TARGET}' was not found."
    )

print(
    f"\nTarget column: {TARGET}"
)

print(
    "\nTarget distribution:"
)

print(
    df[TARGET]
    .value_counts()
)

print(
    "\nTarget percentages:"
)

print(
    (
        df[TARGET]
        .value_counts(normalize=True)
        * 100
    ).round(2)
)

# ------------------------------------------------
# IDENTIFIER / LEAKAGE COLUMNS
# ------------------------------------------------

EXCLUDED_COLUMNS = [
    "post_id",
    "account_id",
    "caption",
    "hashtags",
    "posting_datetime"
]

print("\n" + "=" * 70)
print("EXCLUDED COLUMNS")
print("=" * 70)

for col in EXCLUDED_COLUMNS:

    if col in df.columns:

        print(
            f"EXCLUDE: {col}"
        )

# ------------------------------------------------
# REMAINING FEATURES
# ------------------------------------------------

feature_columns = [
    col
    for col in df.columns
    if (
        col != TARGET
        and col not in EXCLUDED_COLUMNS
    )
]

print("\n" + "=" * 70)
print("REMAINING FEATURES")
print("=" * 70)

print(
    f"\nFeature count: "
    f"{len(feature_columns)}"
)

for i, col in enumerate(
    feature_columns,
    1
):

    print(
        f"{i:02d}. "
        f"{col:<40} "
        f"{df[col].dtype}"
    )

# ------------------------------------------------
# NUMERIC / CATEGORICAL
# ------------------------------------------------

numeric_features = [
    col
    for col in feature_columns
    if pd.api.types.is_numeric_dtype(
        df[col]
    )
]

categorical_features = [
    col
    for col in feature_columns
    if col not in numeric_features
]

print("\n" + "=" * 70)
print("FEATURE TYPES")
print("=" * 70)

print(
    f"\nNumeric features     : "
    f"{len(numeric_features)}"
)

print(
    f"Categorical features : "
    f"{len(categorical_features)}"
)

if categorical_features:

    print(
        "\nCategorical columns:"
    )

    for col in categorical_features:

        print(
            f"  - {col}"
        )

# ------------------------------------------------
# MISSING VALUES
# ------------------------------------------------

print("\n" + "=" * 70)
print("MISSING VALUE AUDIT")
print("=" * 70)

missing = (
    df[feature_columns]
    .isna()
    .sum()
)

missing = missing[
    missing > 0
].sort_values(
    ascending=False
)

if len(missing):

    print(missing)

else:

    print(
        "No missing feature values."
    )

# ------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------

print("\n" + "=" * 70)
print("REBUILD INSPECTION COMPLETED")
print("=" * 70)

print(
    f"""
Dataset:
{DATASET_FILE}

Rows:
{len(df):,}

Target:
{TARGET}

Total candidate features:
{len(feature_columns)}

Numeric features:
{len(numeric_features)}

Categorical features:
{len(categorical_features)}

Next:
REBUILD HISTGRADIENTBOOSTING CHAMPION
"""
)

print("=" * 70)

PRODUCTION MODEL REBUILD

SEARCHING FOR SYNTHETIC V2 DATASET


FileNotFoundError: Synthetic dataset could not be found.

In [4]:
# ================================================================
# 19 - PRODUCTION ML MODEL PIPELINE
# ================================================================

from pathlib import Path
import json
import joblib
import numpy as np

print("=" * 70)
print("PRODUCTION ML MODEL PIPELINE")
print("=" * 70)


# ================================================================
# 1. PROJECT ROOT
# ================================================================

PROJECT_ROOT = Path(
    r"d:\newwwwwwwww\AiBasedInstagramPrediction"
)

print("\nProject root:")
print(PROJECT_ROOT)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root does not exist:\n{PROJECT_ROOT}"
    )


# ================================================================
# 2. EXISTING MODEL FILES
# ================================================================

MODELS_DIR = PROJECT_ROOT / "models"

MODEL_FILE = (
    MODELS_DIR /
    "final_instagram_engagement_model.joblib"
)

FEATURE_FILE = (
    MODELS_DIR /
    "final_model_features.json"
)

METADATA_FILE = (
    MODELS_DIR /
    "final_model_metadata.json"
)


print("\n" + "=" * 70)
print("MODEL FILE VALIDATION")
print("=" * 70)

print(
    "\nModels directory:"
)

print(
    MODELS_DIR
)

print(
    "\nModel:"
)

print(
    MODEL_FILE
)

print(
    "\nFeature file:"
)

print(
    FEATURE_FILE
)

print(
    "\nMetadata file:"
)

print(
    METADATA_FILE
)


# ================================================================
# 3. CHECK FILES
# ================================================================

required_files = [
    MODEL_FILE,
    FEATURE_FILE,
    METADATA_FILE
]

missing_files = [
    str(path)
    for path in required_files
    if not path.exists()
]

if missing_files:

    print("\nMissing files:")

    for path in missing_files:
        print(
            f" - {path}"
        )

    raise FileNotFoundError(
        "\nRequired production model files are missing."
    )


print("\n✓ All production model files found.")


# ================================================================
# 4. LOAD MODEL
# ================================================================

print("\n" + "=" * 70)
print("LOADING FINAL MODEL")
print("=" * 70)

model = joblib.load(
    MODEL_FILE
)

print(
    "\nModel loaded successfully."
)

print(
    "Model type:"
)

print(
    type(model)
)


# ================================================================
# 5. LOAD FEATURE LIST
# ================================================================

print("\n" + "=" * 70)
print("LOADING FEATURE DEFINITIONS")
print("=" * 70)

with open(
    FEATURE_FILE,
    "r",
    encoding="utf-8"
) as f:

    feature_data = json.load(f)

print(
    "\nFeature metadata loaded."
)

print(
    "Feature metadata type:"
)

print(
    type(feature_data)
)


# ================================================================
# 6. DISPLAY FEATURE INFORMATION
# ================================================================

print("\n" + "=" * 70)
print("FEATURE INFORMATION")
print("=" * 70)

if isinstance(
    feature_data,
    list
):

    features = feature_data

elif isinstance(
    feature_data,
    dict
):

    # Try common feature-list keys

    possible_keys = [
        "features",
        "feature_names",
        "columns",
        "model_features",
        "selected_features"
    ]

    features = None

    for key in possible_keys:

        if key in feature_data:

            if isinstance(
                feature_data[key],
                list
            ):

                features = feature_data[key]

                print(
                    f"\nFeature key found: {key}"
                )

                break

    if features is None:

        features = []

        print(
            "\nFeature list key not automatically identified."
        )

else:

    features = []


print(
    f"\nNumber of features: "
    f"{len(features)}"
)

if features:

    print(
        "\nFeatures:"
    )

    for i, feature in enumerate(
        features,
        1
    ):

        print(
            f"{i:03d}. {feature}"
        )


# ================================================================
# 7. LOAD MODEL METADATA
# ================================================================

print("\n" + "=" * 70)
print("LOADING MODEL METADATA")
print("=" * 70)

with open(
    METADATA_FILE,
    "r",
    encoding="utf-8"
) as f:

    metadata = json.load(f)


print(
    "\nMetadata loaded successfully."
)

print(
    "\nMetadata:"
)

print(
    json.dumps(
        metadata,
        indent=4
    )[:15000]
)


# ================================================================
# 8. MODEL INSPECTION
# ================================================================

print("\n" + "=" * 70)
print("MODEL INSPECTION")
print("=" * 70)

print(
    "\nEstimator:"
)

print(
    type(model).__name__
)


if hasattr(
    model,
    "steps"
):

    print(
        "\nPipeline detected."
    )

    print(
        "\nPipeline components:"
    )

    for name, component in model.steps:

        print(
            f" - {name}: "
            f"{type(component).__name__}"
        )

    estimator = model.steps[-1][1]

else:

    estimator = model


print(
    "\nFinal estimator:"
)

print(
    type(estimator).__name__
)


# ================================================================
# 9. CHECK FEATURE COUNT
# ================================================================

print("\n" + "=" * 70)
print("FEATURE COUNT VALIDATION")
print("=" * 70)

model_feature_count = None

if hasattr(
    estimator,
    "n_features_in_"
):

    model_feature_count = (
        estimator.n_features_in_
    )

elif hasattr(
    model,
    "n_features_in_"
):

    model_feature_count = (
        model.n_features_in_
    )


print(
    "\nModel feature count:"
)

print(
    model_feature_count
)

print(
    "\nSaved feature-list count:"
)

print(
    len(features)
)


if (
    model_feature_count is not None
    and len(features) > 0
):

    if model_feature_count == len(features):

        print(
            "\n✓ Feature count matches."
        )

    else:

        print(
            "\n⚠ Feature count mismatch."
        )

        print(
            f"Model expects: "
            f"{model_feature_count}"
        )

        print(
            f"Feature file contains: "
            f"{len(features)}"
        )


# ================================================================
# 10. CHECK CLASSES
# ================================================================

print("\n" + "=" * 70)
print("TARGET CLASSES")
print("=" * 70)

if hasattr(
    estimator,
    "classes_"
):

    classes = estimator.classes_

    print(
        "\nModel classes:"
    )

    print(
        classes
    )

else:

    classes = None

    print(
        "\nClasses are not directly available."
    )


# ================================================================
# 11. PRODUCTION MODEL COPY
# ================================================================

print("\n" + "=" * 70)
print("CREATING PRODUCTION DIRECTORY")
print("=" * 70)

PRODUCTION_DIR = (
    PROJECT_ROOT
    / "models"
    / "production"
)

PRODUCTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


PRODUCTION_MODEL = (
    PRODUCTION_DIR
    / "instagram_engagement_model.joblib"
)

PRODUCTION_FEATURES = (
    PRODUCTION_DIR
    / "instagram_engagement_features.json"
)

PRODUCTION_METADATA = (
    PRODUCTION_DIR
    / "instagram_engagement_metadata.json"
)


# ================================================================
# 12. SAVE PRODUCTION MODEL
# ================================================================

joblib.dump(
    model,
    PRODUCTION_MODEL
)

with open(
    PRODUCTION_FEATURES,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        feature_data,
        f,
        indent=4
    )

with open(
    PRODUCTION_METADATA,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )


print(
    "\n✓ Production model saved."
)

print(
    PRODUCTION_MODEL
)

print(
    "\n✓ Production feature definition saved."
)

print(
    PRODUCTION_FEATURES
)

print(
    "\n✓ Production metadata saved."
)

print(
    PRODUCTION_METADATA
)


# ================================================================
# 13. RELOAD VALIDATION
# ================================================================

print("\n" + "=" * 70)
print("PRODUCTION MODEL RELOAD TEST")
print("=" * 70)

reloaded_model = joblib.load(
    PRODUCTION_MODEL
)

print(
    "\nOriginal model type:"
)

print(
    type(model)
)

print(
    "\nReloaded model type:"
)

print(
    type(reloaded_model)
)


if type(model) == type(
    reloaded_model
):

    print(
        "\n✓ Model reload type check PASSED."
    )

else:

    raise ValueError(
        "Reloaded model type does not match."
    )


# ================================================================
# 14. FINAL STATUS
# ================================================================

print("\n" + "=" * 70)
print("STAGE 1 COMPLETED")
print("=" * 70)

print(
    """
✓ Existing final model located
✓ Final model loaded
✓ Feature definitions loaded
✓ Metadata loaded
✓ Model inspected
✓ Production copy created
✓ Reload test passed

PRIMARY ML MODEL:
HistGradientBoosting

TRAINING FOUNDATION:
Synthetic Instagram Engagement Dataset V2

TARGET:
performance_class

TARGET CLASSES:
Low / Medium / High
"""
)

print("=" * 70)

print(
    "\nREADY FOR STAGE 2"
)

print(
    "IMAGE + TEXT PRODUCTION INTEGRATION"
)

print("=" * 70)

PRODUCTION ML MODEL PIPELINE

Project root:
d:\newwwwwwwww\AiBasedInstagramPrediction

MODEL FILE VALIDATION

Models directory:
d:\newwwwwwwww\AiBasedInstagramPrediction\models

Model:
d:\newwwwwwwww\AiBasedInstagramPrediction\models\final_instagram_engagement_model.joblib

Feature file:
d:\newwwwwwwww\AiBasedInstagramPrediction\models\final_model_features.json

Metadata file:
d:\newwwwwwwww\AiBasedInstagramPrediction\models\final_model_metadata.json

Missing files:
 - d:\newwwwwwwww\AiBasedInstagramPrediction\models\final_instagram_engagement_model.joblib
 - d:\newwwwwwwww\AiBasedInstagramPrediction\models\final_model_features.json
 - d:\newwwwwwwww\AiBasedInstagramPrediction\models\final_model_metadata.json


FileNotFoundError: 
Required production model files are missing.

In [5]:
from pathlib import Path

print("=" * 70)
print("REAL MODEL FILE DISCOVERY")
print("=" * 70)

PROJECT_ROOT = Path(r"d:\newwwwwwwww\AiBasedInstagramPrediction")

print("\nProject root:")
print(PROJECT_ROOT)
print("Exists:", PROJECT_ROOT.exists())

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found:\n{PROJECT_ROOT}"
    )


# ================================================================
# SEARCH ENTIRE PROJECT FOR MODEL FILES
# ================================================================

print("\n" + "=" * 70)
print("SEARCHING ENTIRE PROJECT")
print("=" * 70)

model_extensions = {
    ".joblib",
    ".pkl",
    ".pickle"
}

model_files = []

for path in PROJECT_ROOT.rglob("*"):

    if path.is_file() and path.suffix.lower() in model_extensions:
        model_files.append(path)


print(
    f"\nModel files found: {len(model_files)}"
)


for i, path in enumerate(
    model_files,
    1
):

    print(
        f"{i:02d}. {path}"
    )


# ================================================================
# SEARCH JSON MODEL METADATA
# ================================================================

print("\n" + "=" * 70)
print("SEARCHING MODEL JSON FILES")
print("=" * 70)

json_files = []

for path in PROJECT_ROOT.rglob("*.json"):

    if path.is_file():

        name = path.name.lower()

        if any(
            keyword in name
            for keyword in [
                "model",
                "feature",
                "metadata",
                "engagement"
            ]
        ):

            json_files.append(path)


print(
    f"\nRelevant JSON files found: {len(json_files)}"
)

for i, path in enumerate(
    json_files,
    1
):

    print(
        f"{i:02d}. {path}"
    )


# ================================================================
# SEARCH SYNTHETIC DATASET
# ================================================================

print("\n" + "=" * 70)
print("SEARCHING SYNTHETIC DATASETS")
print("=" * 70)

csv_files = []

for path in PROJECT_ROOT.rglob("*.csv"):

    if path.is_file():

        name = path.name.lower()

        if (
            "synthetic" in name
            or "engagement" in name
            or "instagram" in name
        ):

            csv_files.append(path)


print(
    f"\nRelevant CSV files found: {len(csv_files)}"
)

for i, path in enumerate(
    csv_files,
    1
):

    print(
        f"{i:02d}. {path}"
    )


# ================================================================
# FINAL DIAGNOSTIC
# ================================================================

print("\n" + "=" * 70)
print("DISCOVERY RESULT")
print("=" * 70)

if model_files:

    print(
        "\n✓ MODEL FILES FOUND."
    )

    print(
        "\nWe will use the actual discovered model path."
    )

else:

    print(
        "\n✗ NO JOBLIB / PKL MODEL FILES FOUND."
    )

    print(
        "\nPython cannot currently see the model files."
    )

print(
    "\n" + "=" * 70
)
print("DIAGNOSTIC COMPLETED")
print("=" * 70)

REAL MODEL FILE DISCOVERY

Project root:
d:\newwwwwwwww\AiBasedInstagramPrediction
Exists: True

SEARCHING ENTIRE PROJECT

Model files found: 0

SEARCHING MODEL JSON FILES

Relevant JSON files found: 0

SEARCHING SYNTHETIC DATASETS

Relevant CSV files found: 0

DISCOVERY RESULT

✗ NO JOBLIB / PKL MODEL FILES FOUND.

Python cannot currently see the model files.

DIAGNOSTIC COMPLETED


In [6]:
from pathlib import Path

MODELS_DIR = Path(
    r"D:\newwwwwwww\AiBasedInstagramPrediction\models"
)

print("=" * 70)
print("MODEL DIRECTORY VALIDATION")
print("=" * 70)

print("\nModels directory:")
print(MODELS_DIR)

print("\nExists:", MODELS_DIR.exists())
print("Is directory:", MODELS_DIR.is_dir())

print("\n" + "=" * 70)
print("MODEL FILES")
print("=" * 70)

if MODELS_DIR.exists():

    files = list(MODELS_DIR.iterdir())

    for file in files:
        print(
            "[DIR] " if file.is_dir() else "[FILE]",
            file.name
        )

else:
    raise FileNotFoundError(
        f"Models directory not found:\n{MODELS_DIR}"
    )

print("\n" + "=" * 70)
print("MODEL DISCOVERY COMPLETED")
print("=" * 70)

MODEL DIRECTORY VALIDATION

Models directory:
D:\newwwwwwww\AiBasedInstagramPrediction\models

Exists: True
Is directory: True

MODEL FILES
[FILE] best_combined_model.joblib
[FILE] best_engagement_model.joblib
[FILE] best_model_metadata.json
[FILE] final_instagram_engagement_model.joblib
[FILE] final_model_features.json
[FILE] final_model_metadata.json
[FILE] final_multimodal_or_baseline_model.joblib
[FILE] model_metadata.json
[DIR]  preprocessing

MODEL DISCOVERY COMPLETED


In [7]:
from pathlib import Path
import json

MODELS_DIR = Path(
    r"D:\newwwwww\AiBasedInstagramPrediction\models"
)

print("=" * 70)
print("MODEL METADATA INSPECTION")
print("=" * 70)

json_files = list(MODELS_DIR.glob("*.json"))

for path in json_files:

    print("\n" + "-" * 70)
    print(f"FILE: {path.name}")
    print("-" * 70)

    try:

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)

        print(
            json.dumps(
                data,
                indent=2,
                ensure_ascii=False
            )
        )

    except Exception as e:

        print(
            f"Could not read JSON: {e}"
        )

print("\n" + "=" * 70)
print("METADATA INSPECTION COMPLETED")
print("=" * 70)

MODEL METADATA INSPECTION

METADATA INSPECTION COMPLETED


In [8]:
from pathlib import Path
import json

print("=" * 70)
print("MODEL METADATA INSPECTION")
print("=" * 70)

# This notebook is inside:
# AiBasedInstagramPrediction/notebooks/
# Therefore project root = notebook folder's parent

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    # Fallback: locate project root from the known folder structure
    PROJECT_ROOT = NOTEBOOK_DIR

MODELS_DIR = PROJECT_ROOT / "models"

print("\nCurrent notebook directory:")
print(NOTEBOOK_DIR)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nModels directory:")
print(MODELS_DIR)

print("\nModels directory exists:", MODELS_DIR.exists())


# ================================================================
# DISCOVER ALL JSON FILES
# ================================================================

print("\n" + "=" * 70)
print("JSON FILE DISCOVERY")
print("=" * 70)

json_files = sorted(MODELS_DIR.glob("*.json"))

print(f"\nJSON files found: {len(json_files)}")

for path in json_files:
    print(f"[JSON] {path.name}")


# ================================================================
# INSPECT METADATA
# ================================================================

print("\n" + "=" * 70)
print("MODEL METADATA")
print("=" * 70)

for path in json_files:

    print("\n" + "-" * 70)
    print(f"FILE: {path.name}")
    print("-" * 70)

    try:

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)

        print(
            json.dumps(
                data,
                indent=2,
                ensure_ascii=False
            )
        )

    except Exception as e:

        print(f"ERROR READING FILE: {e}")


print("\n" + "=" * 70)
print("METADATA INSPECTION COMPLETED")
print("=" * 70)

MODEL METADATA INSPECTION

Current notebook directory:
d:\newwwwwwww\AiBasedInstagramPrediction\notebooks

Project root:
d:\newwwwwwww\AiBasedInstagramPrediction

Models directory:
d:\newwwwwwww\AiBasedInstagramPrediction\models

Models directory exists: True

JSON FILE DISCOVERY

JSON files found: 4
[JSON] best_model_metadata.json
[JSON] final_model_features.json
[JSON] final_model_metadata.json
[JSON] model_metadata.json

MODEL METADATA

----------------------------------------------------------------------
FILE: best_model_metadata.json
----------------------------------------------------------------------
{
  "model_name": "Logistic Regression",
  "random_state": 42,
  "training_observations": 81600,
  "feature_count": 1517,
  "target_classes": [
    "Low",
    "Medium",
    "High"
  ],
  "real_test_accuracy": 0.4304,
  "real_test_f1_weighted": 0.4318,
  "preprocessing_reference": "models/preprocessing/preprocessor.pkl"
}

-----------------------------------------------------------

In [9]:
from pathlib import Path
import json

MODELS_DIR = Path.cwd().parent / "models"

files = [
    "best_model_metadata.json",
    "final_model_features.json",
    "final_model_metadata.json",
    "model_metadata.json"
]

print("=" * 70)
print("IMPORTANT MODEL METADATA")
print("=" * 70)

for filename in files:

    path = MODELS_DIR / filename

    print("\n" + "-" * 70)
    print(filename)
    print("-" * 70)

    if not path.exists():
        print("NOT FOUND")
        continue

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Print compactly
        if isinstance(data, dict):

            for key, value in data.items():

                # Avoid dumping enormous feature lists
                if isinstance(value, list) and len(value) > 20:
                    print(f"{key}: [list with {len(value)} items]")
                    print(f"First 10: {value[:10]}")
                else:
                    print(f"{key}: {value}")

        else:
            print(data)

    except Exception as e:
        print("ERROR:", e)

print("\n" + "=" * 70)
print("INSPECTION COMPLETED")
print("=" * 70)

IMPORTANT MODEL METADATA

----------------------------------------------------------------------
best_model_metadata.json
----------------------------------------------------------------------
model_name: Logistic Regression
random_state: 42
training_observations: 81600
feature_count: 1517
target_classes: ['Low', 'Medium', 'High']
real_test_accuracy: 0.4304
real_test_f1_weighted: 0.4318
preprocessing_reference: models/preprocessing/preprocessor.pkl

----------------------------------------------------------------------
final_model_features.json
----------------------------------------------------------------------
numeric_features: [list with 51 items]
First 10: ['follower_count', 'following_count', 'account_age_days', 'verified_status', 'posting_frequency', 'average_historical_engagement', 'audience_growth_rate', 'account_activity_level', 'content_consistency', 'caption_length']
categorical_features: ['category', 'account_type', 'day_of_week', 'posting_time_period', 'media_type']
all_

In [10]:
from pathlib import Path
import json

MODELS_DIR = Path.cwd().parent / "models"

path = MODELS_DIR / "final_model_metadata.json"

print("=" * 70)
print("FINAL MODEL METADATA")
print("=" * 70)

print("\nFile:")
print(path)

print("\nExists:", path.exists())

if not path.exists():
    raise FileNotFoundError(path)

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("\n" + "-" * 70)
print("METADATA")
print("-" * 70)

for key, value in data.items():

    if isinstance(value, list) and len(value) > 20:
        print(f"{key}: [list with {len(value)} items]")
        print(f"First 10: {value[:10]}")
    else:
        print(f"{key}: {value}")

print("\n" + "=" * 70)
print("FINAL MODEL METADATA INSPECTION COMPLETED")
print("=" * 70)

FINAL MODEL METADATA

File:
d:\newwwwwwww\AiBasedInstagramPrediction\models\final_model_metadata.json

Exists: True

----------------------------------------------------------------------
METADATA
----------------------------------------------------------------------
project_name: AI-Based Instagram Engagement Prediction and Content Optimization System
dataset_name: Synthetic V2
dataset_path: d:\newwwwwwww\AiBasedInstagramPrediction\datasets\Synthetic\synthetic_instagram_engagement_dataset_v2.csv
dataset_rows: 100000
dataset_columns: 62
target_column: performance_class
target_classes: ['High', 'Low', 'Medium']
model_name: HistGradientBoostingClassifier
model_parameters: {'learning_rate': 0.08, 'max_iter': 300, 'max_leaf_nodes': 63, 'min_samples_leaf': 20, 'l2_regularization': 0.1, 'random_state': 42}
training_rows: 80000
testing_rows: 20000
feature_count: 56
numeric_feature_count: 51
categorical_feature_count: 5
holdout_accuracy: 0.90965
weighted_precision: 0.9107026696982643
weighted_

In [11]:
from pathlib import Path
import json
import joblib
import numpy as np

print("=" * 70)
print("FINAL PRODUCTION MODEL VERIFICATION")
print("=" * 70)

# ---------------------------------------------------------------
# PROJECT PATH
# ---------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
MODELS_DIR = PROJECT_ROOT / "models"

MODEL_PATH = MODELS_DIR / "final_instagram_engagement_model.joblib"
FEATURE_PATH = MODELS_DIR / "final_model_features.json"
METADATA_PATH = MODELS_DIR / "final_model_metadata.json"

print("\nProject root:")
print(PROJECT_ROOT)

print("\nModel:")
print(MODEL_PATH)

print("\nFeature schema:")
print(FEATURE_PATH)

print("\nMetadata:")
print(METADATA_PATH)


# ---------------------------------------------------------------
# FILE VALIDATION
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("FILE VALIDATION")
print("=" * 70)

for path in [
    MODEL_PATH,
    FEATURE_PATH,
    METADATA_PATH
]:

    print(
        f"{path.name:<40} "
        f"EXISTS = {path.exists()}"
    )

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Final model not found:\n{MODEL_PATH}"
    )

if not FEATURE_PATH.exists():
    raise FileNotFoundError(
        f"Feature schema not found:\n{FEATURE_PATH}"
    )

if not METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Model metadata not found:\n{METADATA_PATH}"
    )


# ---------------------------------------------------------------
# LOAD MODEL
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("LOADING FINAL MODEL")
print("=" * 70)

model = joblib.load(MODEL_PATH)

print("\nModel loaded successfully.")
print("Python class:")
print(type(model))


# ---------------------------------------------------------------
# LOAD FEATURES
# ---------------------------------------------------------------

with open(
    FEATURE_PATH,
    "r",
    encoding="utf-8"
) as f:

    feature_data = json.load(f)


# ---------------------------------------------------------------
# LOAD METADATA
# ---------------------------------------------------------------

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:

    metadata = json.load(f)


# ---------------------------------------------------------------
# MODEL INFORMATION
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL MODEL INFORMATION")
print("=" * 70)

print(
    "\nModel name:",
    metadata.get("model_name")
)

print(
    "Dataset:",
    metadata.get("dataset_name")
)

print(
    "Target:",
    metadata.get("target_column")
)

print(
    "Classes:",
    metadata.get("target_classes")
)

print(
    "Training rows:",
    metadata.get("training_rows")
)

print(
    "Testing rows:",
    metadata.get("testing_rows")
)

print(
    "Feature count:",
    metadata.get("feature_count")
)

print(
    "Numeric feature count:",
    metadata.get("numeric_feature_count")
)


# ---------------------------------------------------------------
# FEATURE SCHEMA
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE SCHEMA")
print("=" * 70)

all_features = feature_data.get(
    "all_features",
    []
)

numeric_features = feature_data.get(
    "numeric_features",
    []
)

categorical_features = feature_data.get(
    "categorical_features",
    []
)

print(
    "\nTotal features:",
    len(all_features)
)

print(
    "Numeric features:",
    len(numeric_features)
)

print(
    "Categorical features:",
    len(categorical_features)
)

print("\nAll features:")

for i, feature in enumerate(
    all_features,
    1
):

    print(
        f"{i:02d}. {feature}"
    )


# ---------------------------------------------------------------
# MODEL INTERNAL FEATURE COUNT
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("MODEL FEATURE VALIDATION")
print("=" * 70)

if hasattr(model, "n_features_in_"):

    print(
        "\nModel expects:",
        model.n_features_in_,
        "features"
    )

    print(
        "Feature schema:",
        len(all_features),
        "features"
    )

    if model.n_features_in_ == len(all_features):

        print(
            "\n✓ FEATURE COUNT MATCHES"
        )

    else:

        print(
            "\n⚠ FEATURE COUNT DOES NOT MATCH"
        )

else:

    print(
        "\nModel does not expose n_features_in_."
    )


# ---------------------------------------------------------------
# CLASS VALIDATION
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASS VALIDATION")
print("=" * 70)

if hasattr(model, "classes_"):

    print(
        "\nModel classes:",
        list(model.classes_)
    )

    print(
        "Metadata classes:",
        metadata.get("target_classes")
    )


# ---------------------------------------------------------------
# FINAL STATUS
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("PRODUCTION MODEL STATUS")
print("=" * 70)

print(
    "\n✓ Final Synthetic V2 model loaded"
)

print(
    "✓ Model metadata loaded"
)

print(
    "✓ Feature schema loaded"
)

print(
    "✓ Ready for production prediction pipeline"
)

print("\n" + "=" * 70)

FINAL PRODUCTION MODEL VERIFICATION

Project root:
d:\newwwwwwww\AiBasedInstagramPrediction

Model:
d:\newwwwwwww\AiBasedInstagramPrediction\models\final_instagram_engagement_model.joblib

Feature schema:
d:\newwwwwwww\AiBasedInstagramPrediction\models\final_model_features.json

Metadata:
d:\newwwwwwww\AiBasedInstagramPrediction\models\final_model_metadata.json

FILE VALIDATION
final_instagram_engagement_model.joblib  EXISTS = True
final_model_features.json                EXISTS = True
final_model_metadata.json                EXISTS = True

LOADING FINAL MODEL

Model loaded successfully.
Python class:
<class 'sklearn.pipeline.Pipeline'>

FINAL MODEL INFORMATION

Model name: HistGradientBoostingClassifier
Dataset: Synthetic V2
Target: performance_class
Classes: ['High', 'Low', 'Medium']
Training rows: 80000
Testing rows: 20000
Feature count: 56
Numeric feature count: 51

FEATURE SCHEMA

Total features: 56
Numeric features: 51
Categorical features: 5

All features:
01. category
02. accou

In [12]:
from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np

print("=" * 70)
print("PRODUCTION FEATURE SCHEMA VALIDATION")
print("=" * 70)

# ---------------------------------------------------------------
# PATHS
# ---------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
MODELS_DIR = PROJECT_ROOT / "models"

MODEL_PATH = MODELS_DIR / "final_instagram_engagement_model.joblib"
FEATURE_PATH = MODELS_DIR / "final_model_features.json"
METADATA_PATH = MODELS_DIR / "final_model_metadata.json"


# ---------------------------------------------------------------
# LOAD MODEL
# ---------------------------------------------------------------

model = joblib.load(MODEL_PATH)

with open(FEATURE_PATH, "r", encoding="utf-8") as f:
    feature_data = json.load(f)

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)


# ---------------------------------------------------------------
# FEATURE LIST
# ---------------------------------------------------------------

all_features = feature_data["all_features"]

numeric_features = feature_data.get(
    "numeric_features",
    []
)

categorical_features = feature_data.get(
    "categorical_features",
    []
)

print("\nModel:")
print(metadata["model_name"])

print("\nTarget:")
print(metadata["target_column"])

print("\nExpected features:", len(all_features))

print(
    "Numeric features:",
    len(numeric_features)
)

print(
    "Categorical features:",
    len(categorical_features)
)


# ---------------------------------------------------------------
# DISPLAY COMPLETE FEATURE SCHEMA
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("COMPLETE 56-FEATURE SCHEMA")
print("=" * 70)

for i, feature in enumerate(all_features, 1):

    if feature in numeric_features:
        feature_type = "NUMERIC"

    elif feature in categorical_features:
        feature_type = "CATEGORICAL"

    else:
        feature_type = "OTHER"

    print(
        f"{i:02d}. {feature:<40} [{feature_type}]"
    )


# ---------------------------------------------------------------
# MODEL EXPECTED FEATURE COUNT
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("MODEL / SCHEMA CONSISTENCY")
print("=" * 70)

if hasattr(model, "n_features_in_"):

    print(
        "\nModel expects:",
        model.n_features_in_
    )

    print(
        "Schema contains:",
        len(all_features)
    )

    if model.n_features_in_ == len(all_features):

        print(
            "\n✓ EXACT FEATURE COUNT MATCH"
        )

    else:

        raise ValueError(
            "Model feature count does not "
            "match saved feature schema."
        )


# ---------------------------------------------------------------
# CHECK DUPLICATES
# ---------------------------------------------------------------

duplicates = (
    pd.Series(all_features)
    .duplicated()
    .sum()
)

print(
    "\nDuplicate features:",
    duplicates
)

if duplicates != 0:

    raise ValueError(
        "Duplicate features found."
    )


# ---------------------------------------------------------------
# FINAL STATUS
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE SCHEMA VALIDATION COMPLETED")
print("=" * 70)

print(
    "\n✓ Production model loaded"
)

print(
    "✓ Feature schema loaded"
)

print(
    "✓ Feature count verified"
)

print(
    "✓ No duplicate features"
)

print(
    "\nNEXT: BUILD USER INPUT FEATURE GENERATOR"
)

print("=" * 70)

PRODUCTION FEATURE SCHEMA VALIDATION

Model:
HistGradientBoostingClassifier

Target:
performance_class

Expected features: 56
Numeric features: 51
Categorical features: 5

COMPLETE 56-FEATURE SCHEMA
01. category                                 [CATEGORICAL]
02. account_type                             [CATEGORICAL]
03. follower_count                           [NUMERIC]
04. following_count                          [NUMERIC]
05. account_age_days                         [NUMERIC]
06. verified_status                          [NUMERIC]
07. posting_frequency                        [NUMERIC]
08. average_historical_engagement            [NUMERIC]
09. audience_growth_rate                     [NUMERIC]
10. account_activity_level                   [NUMERIC]
11. content_consistency                      [NUMERIC]
12. caption_length                           [NUMERIC]
13. word_count                               [NUMERIC]
14. sentence_count                           [NUMERIC]
15. hashtag_count      

In [13]:
# ================================================================
# STAGE 2 — PRODUCTION USER INPUT FEATURE GENERATOR
# ================================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np

print("=" * 70)
print("PRODUCTION USER INPUT FEATURE GENERATOR")
print("=" * 70)


# ---------------------------------------------------------------
# LOAD FEATURE SCHEMA
# ---------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
MODELS_DIR = PROJECT_ROOT / "models"

FEATURE_PATH = MODELS_DIR / "final_model_features.json"

with open(FEATURE_PATH, "r", encoding="utf-8") as f:
    feature_data = json.load(f)

ALL_FEATURES = feature_data["all_features"]

NUMERIC_FEATURES = feature_data.get(
    "numeric_features", []
)

CATEGORICAL_FEATURES = feature_data.get(
    "categorical_features", []
)


# ---------------------------------------------------------------
# DISPLAY SCHEMA
# ---------------------------------------------------------------

print("\nExpected feature count:")
print(len(ALL_FEATURES))

print("\nNumeric features:")
print(len(NUMERIC_FEATURES))

print("\nCategorical features:")
print(len(CATEGORICAL_FEATURES))


# ---------------------------------------------------------------
# GENERIC USER INPUT TEMPLATE
# ---------------------------------------------------------------

user_input = {}

for feature in ALL_FEATURES:

    if feature in NUMERIC_FEATURES:
        user_input[feature] = 0.0

    elif feature in CATEGORICAL_FEATURES:
        user_input[feature] = ""

    else:
        user_input[feature] = 0.0


# ---------------------------------------------------------------
# CREATE DATAFRAME
# ---------------------------------------------------------------

feature_vector = pd.DataFrame(
    [user_input],
    columns=ALL_FEATURES
)


# ---------------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE VECTOR VALIDATION")
print("=" * 70)

print("\nRows:", len(feature_vector))

print(
    "Columns:",
    len(feature_vector.columns)
)


# Check missing columns

missing_columns = [
    feature
    for feature in ALL_FEATURES
    if feature not in feature_vector.columns
]


# Check unexpected columns

unexpected_columns = [
    column
    for column in feature_vector.columns
    if column not in ALL_FEATURES
]


# Check duplicates

duplicate_columns = (
    feature_vector.columns[
        feature_vector.columns.duplicated()
    ].tolist()
)


print("\nMissing columns:")
print(missing_columns)

print("\nUnexpected columns:")
print(unexpected_columns)

print("\nDuplicate columns:")
print(duplicate_columns)


# ---------------------------------------------------------------
# VALIDATION RESULT
# ---------------------------------------------------------------

if missing_columns:
    raise ValueError(
        f"Missing production features: {missing_columns}"
    )

if unexpected_columns:
    raise ValueError(
        f"Unexpected production features: {unexpected_columns}"
    )

if duplicate_columns:
    raise ValueError(
        f"Duplicate production features: {duplicate_columns}"
    )


# ---------------------------------------------------------------
# FEATURE ORDER VALIDATION
# ---------------------------------------------------------------

if list(feature_vector.columns) != ALL_FEATURES:

    raise ValueError(
        "Feature order does not match saved production schema."
    )


print("\n✓ Feature count: 56")
print("✓ Missing features: 0")
print("✓ Unexpected features: 0")
print("✓ Duplicate features: 0")
print("✓ Feature order: VERIFIED")


# ---------------------------------------------------------------
# SAVE TEMPLATE
# ---------------------------------------------------------------

OUTPUT_DIR = PROJECT_ROOT / "results" / "production"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TEMPLATE_PATH = (
    OUTPUT_DIR /
    "production_feature_template.csv"
)

feature_vector.to_csv(
    TEMPLATE_PATH,
    index=False
)


print("\nTemplate saved:")
print(TEMPLATE_PATH)


# ---------------------------------------------------------------
# FINAL STATUS
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("STAGE 2 COMPLETED")
print("=" * 70)

print(
    "\nREADY FOR STAGE 3:"
)

print(
    "REAL USER INPUT → FEATURE ENGINEERING"
)

print("=" * 70)

PRODUCTION USER INPUT FEATURE GENERATOR

Expected feature count:
56

Numeric features:
51

Categorical features:
5

FEATURE VECTOR VALIDATION

Rows: 1
Columns: 56

Missing columns:
[]

Unexpected columns:
[]

Duplicate columns:
[]

✓ Feature count: 56
✓ Missing features: 0
✓ Unexpected features: 0
✓ Duplicate features: 0
✓ Feature order: VERIFIED

Template saved:
d:\newwwwwwww\AiBasedInstagramPrediction\results\production\production_feature_template.csv

STAGE 2 COMPLETED

READY FOR STAGE 3:
REAL USER INPUT → FEATURE ENGINEERING


In [14]:
# ================================================================
# STAGE 3 — CAPTION & HASHTAG FEATURE ENGINEERING
# ================================================================

from pathlib import Path
import json
import re
import math
import pandas as pd
import numpy as np

print("=" * 70)
print("STAGE 3 — CAPTION & HASHTAG FEATURE ENGINEERING")
print("=" * 70)


# ---------------------------------------------------------------
# 1. PROJECT PATHS
# ---------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
MODELS_DIR = PROJECT_ROOT / "models"

FEATURE_SCHEMA_PATH = MODELS_DIR / "final_model_features.json"

if not FEATURE_SCHEMA_PATH.exists():
    raise FileNotFoundError(
        f"Feature schema not found:\n{FEATURE_SCHEMA_PATH}"
    )

print("\nFeature schema:")
print(FEATURE_SCHEMA_PATH)


# ---------------------------------------------------------------
# 2. LOAD PRODUCTION FEATURE SCHEMA
# ---------------------------------------------------------------

with open(FEATURE_SCHEMA_PATH, "r", encoding="utf-8") as f:
    feature_schema = json.load(f)

ALL_FEATURES = feature_schema["all_features"]

print("\nTotal production features:", len(ALL_FEATURES))


# ---------------------------------------------------------------
# 3. TEST USER CONTENT
# ---------------------------------------------------------------

TEST_CAPTION = """
Amazing sunset in Sri Lanka! 🌅✨
Such a beautiful evening by the beach.
#srilanka #travel #sunset #photography #beach
"""

print("\nTest caption:")
print(TEST_CAPTION.strip())


# ---------------------------------------------------------------
# 4. TEXT ENGINEERING FUNCTIONS
# ---------------------------------------------------------------

def clean_caption(text):
    """Basic caption cleaning."""
    
    if text is None:
        return ""
    
    text = str(text)
    
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


def extract_hashtags(text):
    """Extract Instagram hashtags."""
    
    if not text:
        return []
    
    return re.findall(r"(?<!\w)#([A-Za-z0-9_]+)", text)


def count_emojis(text):
    """Approximate emoji count using Unicode ranges."""
    
    if not text:
        return 0
    
    emoji_pattern = re.compile(
        "["
        "\U0001F300-\U0001F5FF"
        "\U0001F600-\U0001F64F"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FAFF"
        "\u2600-\u26FF"
        "\u2700-\u27BF"
        "]",
        flags=re.UNICODE
    )
    
    return len(emoji_pattern.findall(text))


def calculate_sentiment(text):
    """
    Calculate TextBlob sentiment.
    Returns 0 if TextBlob is unavailable.
    """
    
    try:
        from textblob import TextBlob
        
        if not text:
            return 0.0
        
        return float(TextBlob(text).sentiment.polarity)
    
    except Exception:
        return 0.0


def calculate_subjectivity(text):
    """
    Calculate TextBlob subjectivity.
    Returns 0 if unavailable.
    """
    
    try:
        from textblob import TextBlob
        
        if not text:
            return 0.0
        
        return float(TextBlob(text).sentiment.subjectivity)
    
    except Exception:
        return 0.0


# ---------------------------------------------------------------
# 5. EXTRACT TEXT INFORMATION
# ---------------------------------------------------------------

caption = clean_caption(TEST_CAPTION)

hashtags = extract_hashtags(caption)

words_without_hashtags = re.sub(
    r"#\w+",
    "",
    caption
)

words = re.findall(
    r"\b\w+\b",
    words_without_hashtags.lower()
)

caption_length = len(caption)
word_count = len(words)
character_count_no_spaces = len(
    re.sub(r"\s+", "", caption)
)

hashtag_count = len(hashtags)
emoji_count = count_emojis(caption)

sentiment_score = calculate_sentiment(caption)
subjectivity_score = calculate_subjectivity(caption)

uppercase_count = sum(
    1 for c in caption if c.isupper()
)

exclamation_count = caption.count("!")
question_count = caption.count("?")

mention_count = len(
    re.findall(r"(?<!\w)@\w+", caption)
)


# ---------------------------------------------------------------
# 6. DISPLAY EXTRACTED FEATURES
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("EXTRACTED TEXT FEATURES")
print("=" * 70)

text_features = {
    "caption_length": caption_length,
    "word_count": word_count,
    "character_count": character_count_no_spaces,
    "hashtag_count": hashtag_count,
    "emoji_count": emoji_count,
    "sentiment_score": sentiment_score,
    "subjectivity_score": subjectivity_score,
    "uppercase_count": uppercase_count,
    "exclamation_count": exclamation_count,
    "question_count": question_count,
    "mention_count": mention_count,
}


for key, value in text_features.items():
    print(f"{key:<30}: {value}")


print("\nExtracted hashtags:")
print(hashtags)


# ---------------------------------------------------------------
# 7. FIND TEXT-RELATED FEATURES IN THE ACTUAL MODEL SCHEMA
# ---------------------------------------------------------------

TEXT_KEYWORDS = [
    "caption",
    "hashtag",
    "sentiment",
    "emoji",
    "text",
    "keyword",
    "mention",
    "word",
    "character",
    "subjectivity",
    "punctuation"
]


schema_text_features = []

for feature in ALL_FEATURES:
    
    feature_lower = feature.lower()
    
    if any(
        keyword in feature_lower
        for keyword in TEXT_KEYWORDS
    ):
        schema_text_features.append(feature)


print("\n" + "=" * 70)
print("TEXT-RELATED FEATURES FOUND IN PRODUCTION SCHEMA")
print("=" * 70)

if schema_text_features:
    
    for i, feature in enumerate(
        schema_text_features,
        start=1
    ):
        print(f"{i:02d}. {feature}")
        
else:
    print("No obvious text-related features found.")


# ---------------------------------------------------------------
# 8. MAP ENGINEERED VALUES TO MODEL FEATURES
# ---------------------------------------------------------------

feature_values = {}

for feature in ALL_FEATURES:
    
    feature_lower = feature.lower()
    
    value = 0.0
    
    # Caption length
    if feature_lower in [
        "caption_length",
        "caption_len",
        "caption_character_count"
    ]:
        value = caption_length
    
    # Word count
    elif feature_lower in [
        "word_count",
        "caption_word_count",
        "caption_words"
    ]:
        value = word_count
    
    # Hashtags
    elif feature_lower in [
        "hashtag_count",
        "hashtags_count",
        "num_hashtags",
        "hashtag_number"
    ]:
        value = hashtag_count
    
    # Emoji
    elif feature_lower in [
        "emoji_count",
        "emojis_count",
        "num_emojis"
    ]:
        value = emoji_count
    
    # Sentiment
    elif feature_lower in [
        "sentiment",
        "sentiment_score",
        "caption_sentiment",
        "caption_sentiment_score"
    ]:
        value = sentiment_score
    
    # Subjectivity
    elif feature_lower in [
        "subjectivity",
        "subjectivity_score",
        "caption_subjectivity"
    ]:
        value = subjectivity_score
    
    # Mentions
    elif feature_lower in [
        "mention_count",
        "mentions_count",
        "num_mentions"
    ]:
        value = mention_count
    
    # Exclamation
    elif feature_lower in [
        "exclamation_count",
        "exclamation_marks",
        "num_exclamations"
    ]:
        value = exclamation_count
    
    # Question marks
    elif feature_lower in [
        "question_count",
        "question_marks",
        "num_questions"
    ]:
        value = question_count
    
    # Character count
    elif feature_lower in [
        "character_count",
        "char_count",
        "caption_character_count_no_spaces"
    ]:
        value = character_count_no_spaces
    
    feature_values[feature] = value


# ---------------------------------------------------------------
# 9. CREATE PRODUCTION FEATURE ROW
# ---------------------------------------------------------------

production_text_features = pd.DataFrame(
    [feature_values],
    columns=ALL_FEATURES
)


# ---------------------------------------------------------------
# 10. SHOW FEATURES THAT RECEIVED REAL TEXT VALUES
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("MODEL FEATURES POPULATED FROM USER CAPTION")
print("=" * 70)

populated_features = []

for feature in ALL_FEATURES:
    
    value = production_text_features.iloc[0][feature]
    
    if value != 0.0:
        populated_features.append(
            (feature, value)
        )


if populated_features:
    
    for feature, value in populated_features:
        print(
            f"{feature:<40}: {value}"
        )
        
else:
    print(
        "No exact feature-name mappings were activated."
    )


# ---------------------------------------------------------------
# 11. VALIDATE FEATURE VECTOR
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("STAGE 3 FEATURE VECTOR VALIDATION")
print("=" * 70)

print(
    "\nFeature count:",
    len(production_text_features.columns)
)

missing = [
    f for f in ALL_FEATURES
    if f not in production_text_features.columns
]

unexpected = [
    f for f in production_text_features.columns
    if f not in ALL_FEATURES
]

duplicates = (
    production_text_features.columns[
        production_text_features.columns.duplicated()
    ].tolist()
)


print("Missing:", missing)
print("Unexpected:", unexpected)
print("Duplicates:", duplicates)


if missing:
    raise ValueError(
        f"Missing production features: {missing}"
    )

if unexpected:
    raise ValueError(
        f"Unexpected production features: {unexpected}"
    )

if duplicates:
    raise ValueError(
        f"Duplicate features: {duplicates}"
    )


# ---------------------------------------------------------------
# 12. SAVE STAGE 3 OUTPUT
# ---------------------------------------------------------------

OUTPUT_DIR = (
    PROJECT_ROOT /
    "results" /
    "production"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_FILE = (
    OUTPUT_DIR /
    "stage3_caption_hashtag_features.csv"
)

production_text_features.to_csv(
    OUTPUT_FILE,
    index=False
)


# ---------------------------------------------------------------
# FINAL STATUS
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("STAGE 3 COMPLETED")
print("=" * 70)

print(
    "\n✓ Caption processed"
)

print(
    "✓ Hashtags extracted"
)

print(
    "✓ Emoji analysis completed"
)

print(
    "✓ Sentiment analysis completed"
)

print(
    "✓ Text features mapped to production schema"
)

print(
    f"✓ Final feature vector: "
    f"{len(production_text_features.columns)} features"
)

print(
    "\nOutput:"
)

print(OUTPUT_FILE)

print(
    "\nNEXT: STAGE 4 — ACCOUNT & POST FEATURE ENGINEERING"
)

print("=" * 70)

STAGE 3 — CAPTION & HASHTAG FEATURE ENGINEERING

Feature schema:
d:\newwwwwwww\AiBasedInstagramPrediction\models\final_model_features.json

Total production features: 56

Test caption:
Amazing sunset in Sri Lanka! 🌅✨
Such a beautiful evening by the beach.
#srilanka #travel #sunset #photography #beach

EXTRACTED TEXT FEATURES
caption_length                : 116
word_count                    : 12
character_count               : 99
hashtag_count                 : 5
emoji_count                   : 2
sentiment_score               : 0.5333333333333333
subjectivity_score            : 0.7999999999999999
uppercase_count               : 4
exclamation_count             : 1
question_count                : 0
mention_count                 : 0

Extracted hashtags:
['srilanka', 'travel', 'sunset', 'photography', 'beach']

TEXT-RELATED FEATURES FOUND IN PRODUCTION SCHEMA
01. caption_length
02. word_count
03. hashtag_count
04. unique_hashtag_count
05. average_hashtag_length
06. hashtag_character_count
0

In [15]:
# ================================================================
# STAGE 4 — ACCOUNT & POST FEATURE ENGINEERING
# ================================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np
from datetime import datetime

print("=" * 70)
print("STAGE 4 — ACCOUNT & POST FEATURE ENGINEERING")
print("=" * 70)


# ================================================================
# 1. PROJECT PATHS
# ================================================================

PROJECT_ROOT = Path.cwd().parent
MODELS_DIR = PROJECT_ROOT / "models"

FEATURE_SCHEMA_PATH = (
    MODELS_DIR / "final_model_features.json"
)

if not FEATURE_SCHEMA_PATH.exists():
    raise FileNotFoundError(
        f"Feature schema not found:\n{FEATURE_SCHEMA_PATH}"
    )

print("\nFeature schema:")
print(FEATURE_SCHEMA_PATH)


# ================================================================
# 2. LOAD PRODUCTION FEATURE SCHEMA
# ================================================================

with open(
    FEATURE_SCHEMA_PATH,
    "r",
    encoding="utf-8"
) as f:
    feature_schema = json.load(f)

ALL_FEATURES = feature_schema["all_features"]

NUMERIC_FEATURES = feature_schema.get(
    "numeric_features",
    []
)

CATEGORICAL_FEATURES = feature_schema.get(
    "categorical_features",
    []
)

print("\nTotal features:", len(ALL_FEATURES))
print("Numeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))


# ================================================================
# 3. EXAMPLE USER ACCOUNT / POST INPUT
# ================================================================
#
# These are ONLY demonstration values.
# Later these values will come from the React frontend/API.
#

USER_ACCOUNT = {
    "follower_count": 12500,
    "following_count": 850,
    "account_age_days": 1450,
    "verified_status": 0,
    "posting_frequency": 4.0,
    "average_historical_engagement": 0.052,
    "audience_growth_rate": 0.018,
    "account_activity_level": 0.75,
    "content_consistency": 0.70,
}

USER_POST = {
    "category": "Travel",
    "account_type": "Creator",
    "day_of_week": "Saturday",
    "posting_time_period": "Evening",
    "media_type": "Image",
}


# ================================================================
# 4. COMBINE INPUTS
# ================================================================

raw_user_input = {}

raw_user_input.update(USER_ACCOUNT)
raw_user_input.update(USER_POST)


# ================================================================
# 5. FEATURE NAME NORMALIZATION
# ================================================================

def normalize_feature_name(name):
    """
    Normalize feature names so that minor naming differences
    do not break the mapping.
    """
    
    return (
        str(name)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


normalized_input = {
    normalize_feature_name(k): v
    for k, v in raw_user_input.items()
}


# ================================================================
# 6. CREATE COMPLETE 56-FEATURE VECTOR
# ================================================================

feature_values = {}

for feature in ALL_FEATURES:
    
    normalized_feature = normalize_feature_name(feature)
    
    if normalized_feature in normalized_input:
        
        feature_values[feature] = (
            normalized_input[normalized_feature]
        )
    
    else:
        
        # Leave unhandled features at a neutral value.
        if feature in CATEGORICAL_FEATURES:
            feature_values[feature] = ""
        else:
            feature_values[feature] = 0.0


# ================================================================
# 7. CREATE DATAFRAME
# ================================================================

account_post_features = pd.DataFrame(
    [feature_values],
    columns=ALL_FEATURES
)


# ================================================================
# 8. DISPLAY ACCOUNT/POST FEATURES FOUND IN SCHEMA
# ================================================================

ACCOUNT_POST_KEYWORDS = [
    "follower",
    "following",
    "account",
    "posting",
    "historical",
    "engagement",
    "audience",
    "growth",
    "activity",
    "consistency",
    "category",
    "type",
    "day",
    "time",
    "media",
    "verified",
]


schema_account_post_features = []

for feature in ALL_FEATURES:
    
    feature_lower = feature.lower()
    
    if any(
        keyword in feature_lower
        for keyword in ACCOUNT_POST_KEYWORDS
    ):
        schema_account_post_features.append(feature)


print("\n" + "=" * 70)
print("ACCOUNT / POST FEATURES IN PRODUCTION SCHEMA")
print("=" * 70)

for i, feature in enumerate(
    schema_account_post_features,
    start=1
):
    
    value = account_post_features.iloc[0][feature]
    
    print(
        f"{i:02d}. {feature:<40} = {value}"
    )


# ================================================================
# 9. FEATURE TYPE VALIDATION
# ================================================================

print("\n" + "=" * 70)
print("FEATURE TYPE VALIDATION")
print("=" * 70)

numeric_errors = []

for feature in NUMERIC_FEATURES:
    
    value = account_post_features.iloc[0][feature]
    
    if value == "":
        numeric_errors.append(feature)


categorical_errors = []

for feature in CATEGORICAL_FEATURES:
    
    value = account_post_features.iloc[0][feature]
    
    if value is None:
        categorical_errors.append(feature)


print(
    "\nNumeric feature issues:",
    numeric_errors
)

print(
    "Categorical feature issues:",
    categorical_errors
)


# ================================================================
# 10. COMPLETE FEATURE SCHEMA VALIDATION
# ================================================================

print("\n" + "=" * 70)
print("COMPLETE FEATURE VECTOR VALIDATION")
print("=" * 70)

missing_features = [
    feature
    for feature in ALL_FEATURES
    if feature not in account_post_features.columns
]

unexpected_features = [
    feature
    for feature in account_post_features.columns
    if feature not in ALL_FEATURES
]

duplicate_features = (
    account_post_features.columns[
        account_post_features.columns.duplicated()
    ].tolist()
)


print(
    "\nExpected features:",
    len(ALL_FEATURES)
)

print(
    "Actual features:",
    len(account_post_features.columns)
)

print(
    "Missing features:",
    missing_features
)

print(
    "Unexpected features:",
    unexpected_features
)

print(
    "Duplicate features:",
    duplicate_features
)


if missing_features:
    raise ValueError(
        f"Missing production features: {missing_features}"
    )

if unexpected_features:
    raise ValueError(
        f"Unexpected production features: {unexpected_features}"
    )

if duplicate_features:
    raise ValueError(
        f"Duplicate production features: {duplicate_features}"
    )


# ================================================================
# 11. CHECK FEATURE ORDER
# ================================================================

if list(account_post_features.columns) != ALL_FEATURES:
    
    raise ValueError(
        "Feature order does not match production schema."
    )

print("\n✓ Feature order verified")


# ================================================================
# 12. SAVE STAGE 4 OUTPUT
# ================================================================

OUTPUT_DIR = (
    PROJECT_ROOT /
    "results" /
    "production"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_FILE = (
    OUTPUT_DIR /
    "stage4_account_post_features.csv"
)

account_post_features.to_csv(
    OUTPUT_FILE,
    index=False
)


# ================================================================
# 13. FINAL STATUS
# ================================================================

print("\n" + "=" * 70)
print("STAGE 4 COMPLETED")
print("=" * 70)

print("\n✓ Account features prepared")
print("✓ Post features prepared")
print("✓ Categorical values prepared")
print("✓ Numeric values prepared")
print("✓ 56-feature schema preserved")
print("✓ Feature order verified")

print("\nOutput:")
print(OUTPUT_FILE)

print("\nNEXT:")
print("STAGE 5 — IMAGE FEATURE INTEGRATION")

print("=" * 70)

STAGE 4 — ACCOUNT & POST FEATURE ENGINEERING

Feature schema:
d:\newwwwwwww\AiBasedInstagramPrediction\models\final_model_features.json

Total features: 56
Numeric features: 51
Categorical features: 5

ACCOUNT / POST FEATURES IN PRODUCTION SCHEMA
01. category                                 = Travel
02. account_type                             = Creator
03. follower_count                           = 12500
04. following_count                          = 850
05. account_age_days                         = 1450
06. verified_status                          = 0
07. posting_frequency                        = 4.0
08. average_historical_engagement            = 0.052
09. audience_growth_rate                     = 0.018
10. account_activity_level                   = 0.75
11. content_consistency                      = 0.7
12. caption_sentiment                        = 0.0
13. caption_engagement_intent                = 0.0
14. posting_hour                             = 0.0
15. day_of_week           

In [16]:
# ================================================================
# STAGE 5A — IMAGE FEATURE SCHEMA INSPECTION
# ================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

print("=" * 70)
print("STAGE 5A — IMAGE FEATURE SCHEMA INSPECTION")
print("=" * 70)


# ================================================================
# 1. PROJECT PATHS
# ================================================================

PROJECT_ROOT = Path.cwd().parent
MODELS_DIR = PROJECT_ROOT / "models"

FEATURE_SCHEMA_PATH = (
    MODELS_DIR / "final_model_features.json"
)

IMAGE_FEATURE_DIR = (
    PROJECT_ROOT /
    "datasets" /
    "processed" /
    "real_image_features"
)

RESULTS_DIR = (
    PROJECT_ROOT /
    "results" /
    "production"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("\nProject root:")
print(PROJECT_ROOT)

print("\nExpected image feature directory:")
print(IMAGE_FEATURE_DIR)


# ================================================================
# 2. LOAD PRODUCTION FEATURE SCHEMA
# ================================================================

if not FEATURE_SCHEMA_PATH.exists():
    raise FileNotFoundError(
        f"Production feature schema not found:\n"
        f"{FEATURE_SCHEMA_PATH}"
    )

with open(
    FEATURE_SCHEMA_PATH,
    "r",
    encoding="utf-8"
) as f:
    feature_schema = json.load(f)


ALL_FEATURES = feature_schema["all_features"]

NUMERIC_FEATURES = feature_schema.get(
    "numeric_features",
    []
)

CATEGORICAL_FEATURES = feature_schema.get(
    "categorical_features",
    []
)


print("\n" + "=" * 70)
print("PRODUCTION MODEL FEATURE SCHEMA")
print("=" * 70)

print("\nTotal features:", len(ALL_FEATURES))
print("Numeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))


# ================================================================
# 3. IDENTIFY POSSIBLE IMAGE FEATURES IN MODEL SCHEMA
# ================================================================

IMAGE_KEYWORDS = [
    "image",
    "visual",
    "img",
    "brightness",
    "contrast",
    "sharpness",
    "color",
    "saturation",
    "composition",
    "resolution",
    "aspect",
    "photo",
    "vision",
    "resnet",
    "embedding"
]


image_schema_features = []

for feature in ALL_FEATURES:

    feature_lower = str(feature).lower()

    if any(
        keyword in feature_lower
        for keyword in IMAGE_KEYWORDS
    ):
        image_schema_features.append(feature)


print("\n" + "=" * 70)
print("IMAGE-RELATED FEATURES IN PRODUCTION SCHEMA")
print("=" * 70)

if image_schema_features:

    for i, feature in enumerate(
        image_schema_features,
        start=1
    ):
        print(f"{i:02d}. {feature}")

else:

    print(
        "No explicit image/visual features "
        "were found in the 56-feature production schema."
    )


# ================================================================
# 4. DISCOVER EXISTING IMAGE FEATURE FILES
# ================================================================

print("\n" + "=" * 70)
print("IMAGE FEATURE FILE DISCOVERY")
print("=" * 70)


possible_feature_files = []

search_directories = [
    IMAGE_FEATURE_DIR,
    PROJECT_ROOT / "results",
    PROJECT_ROOT / "results" / "deep_image_features",
    PROJECT_ROOT / "results" / "real_image_analysis",
    PROJECT_ROOT / "datasets" / "processed"
]


for directory in search_directories:

    if directory.exists():

        for file_path in directory.rglob("*"):

            if file_path.is_file():

                if file_path.suffix.lower() in [
                    ".npy",
                    ".npz",
                    ".csv"
                ]:

                    if file_path not in possible_feature_files:
                        possible_feature_files.append(
                            file_path
                        )


print(
    "\nCandidate image feature files found:",
    len(possible_feature_files)
)


for i, file_path in enumerate(
    possible_feature_files,
    start=1
):

    print(
        f"{i:02d}. {file_path}"
    )


# ================================================================
# 5. INSPECT NPY FILES
# ================================================================

print("\n" + "=" * 70)
print("NUMPY IMAGE FEATURE INSPECTION")
print("=" * 70)


npy_results = []


for file_path in possible_feature_files:

    if file_path.suffix.lower() != ".npy":
        continue

    try:

        array = np.load(
            file_path,
            mmap_mode="r"
        )

        shape = array.shape
        dtype = array.dtype

        npy_results.append({
            "file": str(file_path),
            "shape": shape,
            "dtype": str(dtype)
        })

        print("\nFILE:")
        print(file_path)

        print("Shape:", shape)
        print("Dtype:", dtype)

        if len(shape) == 2:

            print(
                "Samples:",
                shape[0]
            )

            print(
                "Features:",
                shape[1]
            )

    except Exception as e:

        print(
            f"\nCould not inspect {file_path}"
        )

        print(
            "Reason:",
            str(e)
        )


# ================================================================
# 6. INSPECT CSV IMAGE FEATURE FILES
# ================================================================

print("\n" + "=" * 70)
print("CSV IMAGE FEATURE INSPECTION")
print("=" * 70)


csv_results = []


for file_path in possible_feature_files:

    if file_path.suffix.lower() != ".csv":
        continue

    try:

        # Only inspect structure.
        df = pd.read_csv(
            file_path,
            nrows=5
        )

        csv_results.append({
            "file": str(file_path),
            "columns": len(df.columns),
            "rows_sampled": len(df)
        })

        print("\nFILE:")
        print(file_path)

        print(
            "Columns:",
            len(df.columns)
        )

        print(
            "Sample rows:",
            len(df)
        )

        print(
            "First columns:"
        )

        for column in df.columns[:15]:

            print(
                " -",
                column
            )

    except Exception as e:

        print(
            f"\nCould not inspect {file_path}"
        )

        print(
            "Reason:",
            str(e)
        )


# ================================================================
# 7. CHECK FOR RESNET-50 2048-D FEATURE REPRESENTATION
# ================================================================

print("\n" + "=" * 70)
print("RESNET-50 REPRESENTATION CHECK")
print("=" * 70)


resnet_candidates = []

for result in npy_results:

    shape = result["shape"]

    if (
        len(shape) == 2
        and shape[1] == 2048
    ):

        resnet_candidates.append(
            result
        )


for result in csv_results:

    if result["columns"] == 2048:

        resnet_candidates.append(
            result
        )


if resnet_candidates:

    print(
        "\n✓ 2048-dimensional image representation detected."
    )

    for result in resnet_candidates:

        print(
            "\nCandidate:",
            result["file"]
        )

        print(
            "Shape:",
            result.get("shape", "CSV")
        )

else:

    print(
        "\nNo 2048-dimensional ResNet representation "
        "was automatically detected."
    )


# ================================================================
# 8. COMPATIBILITY ANALYSIS
# ================================================================

print("\n" + "=" * 70)
print("IMAGE / PRODUCTION MODEL COMPATIBILITY")
print("=" * 70)


print(
    "\nProduction model input:",
    len(ALL_FEATURES),
    "features"
)

print(
    "Available ResNet representation:",
    "2048 features"
)


if image_schema_features:

    print(
        "\nExplicit image features exist in the "
        "production schema."
    )

    print(
        "These features must be mapped carefully "
        "before production prediction."
    )

else:

    print(
        "\nNo explicit ResNet/image feature columns "
        "exist in the 56-feature production schema."
    )

    print(
        "Therefore, the 2048 ResNet features cannot "
        "be directly appended to the production model."
    )


# ================================================================
# 9. SAVE INSPECTION REPORT
# ================================================================

inspection_report = {

    "production_feature_count": len(
        ALL_FEATURES
    ),

    "numeric_feature_count": len(
        NUMERIC_FEATURES
    ),

    "categorical_feature_count": len(
        CATEGORICAL_FEATURES
    ),

    "image_schema_features": (
        image_schema_features
    ),

    "image_feature_candidates": [
        {
            "file": result["file"],
            "shape": result.get(
                "shape",
                None
            ),
            "dtype": result.get(
                "dtype",
                None
            )
        }
        for result in npy_results
    ],

    "resnet_2048_candidates": [
        result["file"]
        for result in resnet_candidates
    ]
}


REPORT_PATH = (
    RESULTS_DIR /
    "stage5a_image_feature_schema_inspection.json"
)


with open(
    REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        inspection_report,
        f,
        indent=4
    )


print("\nInspection report saved:")
print(REPORT_PATH)


# ================================================================
# FINAL STATUS
# ================================================================

print("\n" + "=" * 70)
print("STAGE 5A COMPLETED")
print("=" * 70)

print(
    "\n✓ Production schema inspected"
)

print(
    "✓ Image-related production features checked"
)

print(
    "✓ Existing image feature files searched"
)

print(
    "✓ ResNet representation checked"
)

print(
    "✓ Compatibility analysis completed"
)

print(
    "\nNEXT:"
)

print(
    "STAGE 5B — IMAGE FEATURE INTEGRATION"
)

print("=" * 70)

STAGE 5A — IMAGE FEATURE SCHEMA INSPECTION

Project root:
d:\newwwwwwww\AiBasedInstagramPrediction

Expected image feature directory:
d:\newwwwwwww\AiBasedInstagramPrediction\datasets\processed\real_image_features

PRODUCTION MODEL FEATURE SCHEMA

Total features: 56
Numeric features: 51
Categorical features: 5

IMAGE-RELATED FEATURES IN PRODUCTION SCHEMA
01. has_image
02. image_width
03. image_height
04. aspect_ratio
05. brightness
06. contrast
07. saturation
08. sharpness
09. colorfulness
10. text_in_image
11. visual_complexity
12. estimated_image_quality

IMAGE FEATURE FILE DISCOVERY

Candidate image feature files found: 87
01. d:\newwwwwwww\AiBasedInstagramPrediction\datasets\processed\real_image_features\real_resnet50_features.npy
02. d:\newwwwwwww\AiBasedInstagramPrediction\datasets\processed\real_image_features\real_resnet50_metadata.csv
03. d:\newwwwwwww\AiBasedInstagramPrediction\results\baseline_vs_image_enhanced.csv
04. d:\newwwwwwww\AiBasedInstagramPrediction\results\best_mo

In [17]:
# ================================================================
# STAGE 5B — IMAGE FEATURE INTEGRATION
# ================================================================

from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

print("=" * 70)
print("STAGE 5B — IMAGE FEATURE INTEGRATION")
print("=" * 70)


# ================================================================
# 1. PROJECT PATHS
# ================================================================

PROJECT_ROOT = Path.cwd().parent
MODELS_DIR = PROJECT_ROOT / "models"

FEATURE_SCHEMA_PATH = (
    MODELS_DIR / "final_model_features.json"
)

RAW_DATA_DIR = (
    PROJECT_ROOT / "datasets" / "raw"
)

OUTPUT_DIR = (
    PROJECT_ROOT /
    "results" /
    "production"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 2. LOAD PRODUCTION FEATURE SCHEMA
# ================================================================

if not FEATURE_SCHEMA_PATH.exists():
    raise FileNotFoundError(
        f"Feature schema not found:\n{FEATURE_SCHEMA_PATH}"
    )

with open(
    FEATURE_SCHEMA_PATH,
    "r",
    encoding="utf-8"
) as f:
    schema = json.load(f)

ALL_FEATURES = schema["all_features"]

NUMERIC_FEATURES = schema.get(
    "numeric_features",
    []
)

CATEGORICAL_FEATURES = schema.get(
    "categorical_features",
    []
)


print("\nProduction features:", len(ALL_FEATURES))
print("Numeric:", len(NUMERIC_FEATURES))
print("Categorical:", len(CATEGORICAL_FEATURES))


# ================================================================
# 3. FIND A REAL TEST IMAGE
# ================================================================

print("\n" + "=" * 70)
print("SEARCHING FOR REAL TEST IMAGE")
print("=" * 70)

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
    ".bmp"
}

image_candidates = []

if RAW_DATA_DIR.exists():

    for path in RAW_DATA_DIR.rglob("*"):

        if (
            path.is_file()
            and path.suffix.lower()
            in image_extensions
        ):
            image_candidates.append(path)


if not image_candidates:

    raise FileNotFoundError(
        "No real image could be found in the raw dataset."
    )


TEST_IMAGE = image_candidates[0]

print("\nTest image:")
print(TEST_IMAGE)


# ================================================================
# 4. LOAD IMAGE
# ================================================================

try:

    from PIL import Image

except ImportError:

    raise ImportError(
        "Pillow is required for Stage 5B."
    )


image = Image.open(TEST_IMAGE).convert("RGB")

width, height = image.size

print("\nImage loaded successfully.")
print("Width :", width)
print("Height:", height)


# ================================================================
# 5. BASIC IMAGE FEATURES
# ================================================================

image_array = np.asarray(
    image,
    dtype=np.float32
)

gray = (
    0.299 * image_array[:, :, 0]
    + 0.587 * image_array[:, :, 1]
    + 0.114 * image_array[:, :, 2]
)


# Brightness
brightness = float(
    np.mean(gray)
)


# Contrast
contrast = float(
    np.std(gray)
)


# Aspect ratio
aspect_ratio = (
    float(width) / float(height)
    if height > 0
    else 0.0
)


# Color statistics
mean_red = float(
    np.mean(image_array[:, :, 0])
)

mean_green = float(
    np.mean(image_array[:, :, 1])
)

mean_blue = float(
    np.mean(image_array[:, :, 2])
)


std_red = float(
    np.std(image_array[:, :, 0])
)

std_green = float(
    np.std(image_array[:, :, 1])
)

std_blue = float(
    np.std(image_array[:, :, 2])
)


# ================================================================
# 6. SATURATION
# ================================================================

normalized = image_array / 255.0

max_channel = np.max(
    normalized,
    axis=2
)

min_channel = np.min(
    normalized,
    axis=2
)

saturation = np.where(
    max_channel == 0,
    0,
    (max_channel - min_channel) / max_channel
)

mean_saturation = float(
    np.mean(saturation)
)


# ================================================================
# 7. SHARPNESS ESTIMATION
# ================================================================

# Simple gradient-based sharpness measure.

gradient_x = np.diff(
    gray,
    axis=1
)

gradient_y = np.diff(
    gray,
    axis=0
)

sharpness = float(
    (
        np.var(gradient_x)
        +
        np.var(gradient_y)
    ) / 2
)


# ================================================================
# 8. IMAGE FEATURE DICTIONARY
# ================================================================

raw_image_features = {

    "has_image": 1,

    "image_width": width,

    "image_height": height,

    "aspect_ratio": aspect_ratio,

    "brightness": brightness,

    "average_brightness": brightness,

    "image_brightness": brightness,

    "contrast": contrast,

    "image_contrast": contrast,

    "sharpness": sharpness,

    "image_sharpness": sharpness,

    "saturation": mean_saturation,

    "color_saturation": mean_saturation,

    "mean_red": mean_red,

    "mean_green": mean_green,

    "mean_blue": mean_blue,

    "red_mean": mean_red,

    "green_mean": mean_green,

    "blue_mean": mean_blue,

    "red_std": std_red,

    "green_std": std_green,

    "blue_std": std_blue,

    "color_variance": float(
        np.var(image_array)
    ),
}


# ================================================================
# 9. NORMALIZE FEATURE NAMES
# ================================================================

def normalize_name(name):

    return (
        str(name)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


normalized_image_features = {
    normalize_name(k): v
    for k, v in raw_image_features.items()
}


# ================================================================
# 10. MAP ONLY FEATURES ACTUALLY EXPECTED BY MODEL
# ================================================================

image_feature_values = {}

matched_features = []

unmatched_image_features = []


for feature in ALL_FEATURES:

    normalized_feature = normalize_name(
        feature
    )

    if normalized_feature in normalized_image_features:

        value = normalized_image_features[
            normalized_feature
        ]

        image_feature_values[feature] = value

        matched_features.append(
            feature
        )

    else:

        # Keep categorical features empty.
        # Keep numeric features neutral.
        if feature in CATEGORICAL_FEATURES:

            image_feature_values[feature] = ""

        else:

            image_feature_values[feature] = 0.0


# ================================================================
# 11. DISPLAY IMAGE FEATURES FOUND
# ================================================================

print("\n" + "=" * 70)
print("IMAGE FEATURES USED BY PRODUCTION SCHEMA")
print("=" * 70)

for feature in matched_features:

    value = image_feature_values[
        feature
    ]

    print(
        f"{feature:<40} = {value}"
    )


# ================================================================
# 12. IMAGE FEATURE SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("IMAGE ANALYSIS SUMMARY")
print("=" * 70)

print(
    f"\nImage dimensions : {width} x {height}"
)

print(
    f"Aspect ratio     : {aspect_ratio:.4f}"
)

print(
    f"Brightness       : {brightness:.4f}"
)

print(
    f"Contrast         : {contrast:.4f}"
)

print(
    f"Sharpness        : {sharpness:.4f}"
)

print(
    f"Saturation       : {mean_saturation:.4f}"
)


# ================================================================
# 13. CREATE 56-FEATURE IMAGE VECTOR
# ================================================================

image_feature_vector = pd.DataFrame(
    [image_feature_values],
    columns=ALL_FEATURES
)


# ================================================================
# 14. VALIDATION
# ================================================================

print("\n" + "=" * 70)
print("IMAGE FEATURE VECTOR VALIDATION")
print("=" * 70)

missing = [
    feature
    for feature in ALL_FEATURES
    if feature not in image_feature_vector.columns
]

unexpected = [
    feature
    for feature in image_feature_vector.columns
    if feature not in ALL_FEATURES
]

duplicates = (
    image_feature_vector.columns[
        image_feature_vector.columns.duplicated()
    ].tolist()
)


print(
    "\nExpected features:",
    len(ALL_FEATURES)
)

print(
    "Actual features:",
    len(image_feature_vector.columns)
)

print(
    "Missing:",
    missing
)

print(
    "Unexpected:",
    unexpected
)

print(
    "Duplicates:",
    duplicates
)


if missing:

    raise ValueError(
        f"Missing production features: {missing}"
    )


if unexpected:

    raise ValueError(
        f"Unexpected production features: {unexpected}"
    )


if duplicates:

    raise ValueError(
        f"Duplicate production features: {duplicates}"
    )


if list(
    image_feature_vector.columns
) != ALL_FEATURES:

    raise ValueError(
        "Image feature vector order does not "
        "match production schema."
    )


# ================================================================
# 15. SAVE IMAGE FEATURE VECTOR
# ================================================================

OUTPUT_FILE = (
    OUTPUT_DIR /
    "stage5b_image_feature_vector.csv"
)

image_feature_vector.to_csv(
    OUTPUT_FILE,
    index=False
)


# ================================================================
# 16. SAVE IMAGE ANALYSIS METADATA
# ================================================================

image_metadata = {

    "image_path": str(TEST_IMAGE),

    "width": width,

    "height": height,

    "aspect_ratio": aspect_ratio,

    "brightness": brightness,

    "contrast": contrast,

    "sharpness": sharpness,

    "saturation": mean_saturation,

    "matched_production_features":
        matched_features,

    "production_feature_count":
        len(ALL_FEATURES),

    "matched_feature_count":
        len(matched_features)
}


METADATA_FILE = (
    OUTPUT_DIR /
    "stage5b_image_metadata.json"
)


with open(
    METADATA_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        image_metadata,
        f,
        indent=4
    )


# ================================================================
# FINAL STATUS
# ================================================================

print("\n" + "=" * 70)
print("STAGE 5B COMPLETED")
print("=" * 70)

print(
    "\n✓ Real Instagram image loaded"
)

print(
    "✓ Image dimensions extracted"
)

print(
    "✓ Aspect ratio extracted"
)

print(
    "✓ Brightness extracted"
)

print(
    "✓ Contrast extracted"
)

print(
    "✓ Sharpness extracted"
)

print(
    "✓ Colour/saturation features extracted"
)

print(
    "✓ Features mapped to production schema"
)

print(
    f"✓ Production vector contains "
    f"{len(image_feature_vector.columns)} features"
)

print(
    "\nImage feature vector:"
)

print(
    OUTPUT_FILE
)

print(
    "\nMetadata:"
)

print(
    METADATA_FILE
)

print(
    "\nNEXT:"
)

print(
    "STAGE 6 — MULTIMODAL 56-FEATURE ASSEMBLY"
)

print("=" * 70)

STAGE 5B — IMAGE FEATURE INTEGRATION

Production features: 56
Numeric: 51
Categorical: 5

SEARCHING FOR REAL TEST IMAGE

Test image:
d:\newwwwwwww\AiBasedInstagramPrediction\datasets\raw\instagram_data\img\insta1.jpg

Image loaded successfully.
Width : 1080
Height: 1080

IMAGE FEATURES USED BY PRODUCTION SCHEMA
has_image                                = 1
image_width                              = 1080
image_height                             = 1080
aspect_ratio                             = 1.0
brightness                               = 116.99628448486328
contrast                                 = 68.01126098632812
saturation                               = 0.3789421021938324
sharpness                                = 179.89495849609375

IMAGE ANALYSIS SUMMARY

Image dimensions : 1080 x 1080
Aspect ratio     : 1.0000
Brightness       : 116.9963
Contrast         : 68.0113
Sharpness        : 179.8950
Saturation       : 0.3789

IMAGE FEATURE VECTOR VALIDATION

Expected features: 56
Actua

In [18]:
# ================================================================
# STAGE 6 — MULTIMODAL 56-FEATURE ASSEMBLY
# ================================================================

from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np

print("=" * 70)
print("STAGE 6 — MULTIMODAL 56-FEATURE ASSEMBLY")
print("=" * 70)


# ================================================================
# 1. PROJECT PATHS
# ================================================================

PROJECT_ROOT = Path.cwd().parent
MODELS_DIR = PROJECT_ROOT / "models"
PRODUCTION_RESULTS = (
    PROJECT_ROOT / "results" / "production"
)

PRODUCTION_RESULTS.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_PATH = (
    MODELS_DIR /
    "final_instagram_engagement_model.joblib"
)

FEATURE_SCHEMA_PATH = (
    MODELS_DIR /
    "final_model_features.json"
)

METADATA_PATH = (
    MODELS_DIR /
    "final_model_metadata.json"
)


# ================================================================
# 2. LOAD PRODUCTION MODEL + SCHEMA
# ================================================================

print("\n" + "=" * 70)
print("LOADING PRODUCTION COMPONENTS")
print("=" * 70)

model = joblib.load(
    MODEL_PATH
)

with open(
    FEATURE_SCHEMA_PATH,
    "r",
    encoding="utf-8"
) as f:
    schema = json.load(f)

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:
    metadata = json.load(f)


ALL_FEATURES = schema["all_features"]

NUMERIC_FEATURES = schema.get(
    "numeric_features",
    []
)

CATEGORICAL_FEATURES = schema.get(
    "categorical_features",
    []
)


print("\nModel:")
print(type(model).__name__)

print(
    "Expected features:",
    len(ALL_FEATURES)
)

print(
    "Numeric:",
    len(NUMERIC_FEATURES)
)

print(
    "Categorical:",
    len(CATEGORICAL_FEATURES)
)


# ================================================================
# 3. LOAD STAGE 3 OUTPUT
# ================================================================

print("\n" + "=" * 70)
print("LOADING STAGE 3 — TEXT FEATURES")
print("=" * 70)

STAGE3_FILE = (
    PRODUCTION_RESULTS /
    "stage3_caption_hashtag_features.csv"
)

if not STAGE3_FILE.exists():
    raise FileNotFoundError(
        f"Stage 3 output not found:\n{STAGE3_FILE}"
    )

stage3 = pd.read_csv(
    STAGE3_FILE
)

print(
    "\nStage 3 shape:",
    stage3.shape
)


# ================================================================
# 4. LOAD STAGE 4 OUTPUT
# ================================================================

print("\n" + "=" * 70)
print("LOADING STAGE 4 — ACCOUNT / POST FEATURES")
print("=" * 70)

STAGE4_FILE = (
    PRODUCTION_RESULTS /
    "stage4_account_post_features.csv"
)

if not STAGE4_FILE.exists():
    raise FileNotFoundError(
        f"Stage 4 output not found:\n{STAGE4_FILE}"
    )

stage4 = pd.read_csv(
    STAGE4_FILE
)

print(
    "\nStage 4 shape:",
    stage4.shape
)


# ================================================================
# 5. LOAD STAGE 5B OUTPUT
# ================================================================

print("\n" + "=" * 70)
print("LOADING STAGE 5B — IMAGE FEATURES")
print("=" * 70)

STAGE5_FILE = (
    PRODUCTION_RESULTS /
    "stage5b_image_feature_vector.csv"
)

if not STAGE5_FILE.exists():
    raise FileNotFoundError(
        f"Stage 5B output not found:\n{STAGE5_FILE}"
    )

stage5 = pd.read_csv(
    STAGE5_FILE
)

print(
    "\nStage 5B shape:",
    stage5.shape
)


# ================================================================
# 6. VALIDATE INDIVIDUAL STAGE OUTPUTS
# ================================================================

print("\n" + "=" * 70)
print("INDIVIDUAL FEATURE OUTPUT VALIDATION")
print("=" * 70)

for name, df in [
    ("Stage 3", stage3),
    ("Stage 4", stage4),
    ("Stage 5B", stage5)
]:

    missing = [
        feature
        for feature in ALL_FEATURES
        if feature not in df.columns
    ]

    unexpected = [
        column
        for column in df.columns
        if column not in ALL_FEATURES
    ]

    print(
        f"\n{name}:"
    )

    print(
        "  Rows:",
        len(df)
    )

    print(
        "  Columns:",
        len(df.columns)
    )

    print(
        "  Missing:",
        len(missing)
    )

    print(
        "  Unexpected:",
        len(unexpected)
    )


# ================================================================
# 7. CREATE EMPTY MASTER VECTOR
# ================================================================

master = pd.DataFrame(
    np.nan,
    index=[0],
    columns=ALL_FEATURES
)


# ================================================================
# 8. MERGE FEATURES WITHOUT DUPLICATING COLUMNS
# ================================================================

# Stage 3 contains text-derived values.
# Stage 4 contains account/post values.
# Stage 5B contains image values.

source_frames = [
    ("Stage 3", stage3),
    ("Stage 4", stage4),
    ("Stage 5B", stage5)
]


print("\n" + "=" * 70)
print("MERGING MULTIMODAL FEATURES")
print("=" * 70)


for source_name, source_df in source_frames:

    for feature in ALL_FEATURES:

        if feature not in source_df.columns:
            continue

        source_value = source_df.iloc[0][feature]

        # Treat empty strings and NaN as unavailable.
        is_missing = (
            pd.isna(source_value)
            or (
                isinstance(
                    source_value,
                    str
                )
                and source_value.strip() == ""
            )
        )

        if is_missing:
            continue

        current_value = master.iloc[0][feature]

        current_missing = pd.isna(
            current_value
        )

        if current_missing:

            master.loc[0, feature] = source_value

        else:

            # If both stages populated the same feature,
            # check whether they agree.

            try:

                same_value = (
                    float(current_value)
                    == float(source_value)
                )

            except Exception:

                same_value = (
                    str(current_value)
                    == str(source_value)
                )

            if not same_value:

                print(
                    f"WARNING: conflicting value for "
                    f"'{feature}'"
                )

                print(
                    f"  Existing: {current_value}"
                )

                print(
                    f"  New ({source_name}): "
                    f"{source_value}"
                )

                # Keep the most specific later-stage
                # value. Stage 5B is the latest stage.
                master.loc[0, feature] = source_value


# ================================================================
# 9. FILL FEATURES NOT YET PROVIDED
# ================================================================

print("\n" + "=" * 70)
print("UNPOPULATED FEATURE CHECK")
print("=" * 70)

unpopulated = []

for feature in ALL_FEATURES:

    value = master.iloc[0][feature]

    if pd.isna(value):

        unpopulated.append(
            feature
        )


print(
    "\nUnpopulated features:",
    len(unpopulated)
)

for feature in unpopulated:
    print(
        " -",
        feature
    )


# IMPORTANT:
# We do NOT silently invent values for these features.
# They must be handled before production prediction.

if unpopulated:

    print(
        "\n⚠ Some production features are still "
        "not populated."
    )

    print(
        "This is expected at this assembly stage."
    )

    print(
        "We will resolve these before calling "
        "the final model."
    )


# ================================================================
# 10. FEATURE ORDER VALIDATION
# ================================================================

print("\n" + "=" * 70)
print("FEATURE ORDER VALIDATION")
print("=" * 70)

if list(master.columns) != ALL_FEATURES:

    raise ValueError(
        "Master feature order does not match "
        "production schema."
    )

print(
    "\n✓ Feature order exactly matches schema."
)


# ================================================================
# 11. FEATURE COUNT VALIDATION
# ================================================================

print("\n" + "=" * 70)
print("FEATURE COUNT VALIDATION")
print("=" * 70)

print(
    "\nExpected:",
    len(ALL_FEATURES)
)

print(
    "Actual:",
    len(master.columns)
)

if len(master.columns) != 56:

    raise ValueError(
        "Production feature count is not 56."
    )

print(
    "\n✓ Exactly 56 feature columns assembled."
)


# ================================================================
# 12. DATA TYPE CHECK
# ================================================================

print("\n" + "=" * 70)
print("DATA TYPE CHECK")
print("=" * 70)

numeric_type_errors = []

for feature in NUMERIC_FEATURES:

    value = master.iloc[0][feature]

    if pd.isna(value):
        continue

    try:

        float(value)

    except Exception:

        numeric_type_errors.append(
            (feature, value)
        )


print(
    "\nNumeric type errors:",
    len(numeric_type_errors)
)

for feature, value in numeric_type_errors:

    print(
        f" - {feature}: {value}"
    )


if numeric_type_errors:

    raise TypeError(
        "One or more numeric features contain "
        "non-numeric values."
    )


# ================================================================
# 13. CONVERT NUMERIC FEATURES
# ================================================================

for feature in NUMERIC_FEATURES:

    master[feature] = pd.to_numeric(
        master[feature],
        errors="coerce"
    )


# ================================================================
# 14. DISPLAY FINAL MULTIMODAL VECTOR
# ================================================================

print("\n" + "=" * 70)
print("FINAL MULTIMODAL FEATURE VECTOR")
print("=" * 70)

for i, feature in enumerate(
    ALL_FEATURES,
    start=1
):

    value = master.iloc[0][feature]

    print(
        f"{i:02d}. "
        f"{feature:<40} = {value}"
    )


# ================================================================
# 15. SAVE ASSEMBLED VECTOR
# ================================================================

ASSEMBLED_FILE = (
    PRODUCTION_RESULTS /
    "stage6_multimodal_56_feature_vector.csv"
)

master.to_csv(
    ASSEMBLED_FILE,
    index=False
)


print("\n" + "=" * 70)
print("ASSEMBLED VECTOR SAVED")
print("=" * 70)

print(
    ASSEMBLED_FILE
)


# ================================================================
# 16. FINAL STAGE 6 STATUS
# ================================================================

print("\n" + "=" * 70)
print("STAGE 6 COMPLETED — ASSEMBLY")
print("=" * 70)

print(
    "\n✓ Stage 3 text features loaded"
)

print(
    "✓ Stage 4 account/post features loaded"
)

print(
    "✓ Stage 5B image features loaded"
)

print(
    "✓ Feature sources merged"
)

print(
    "✓ Exactly 56 feature columns maintained"
)

print(
    "✓ Feature order verified"
)

print(
    "✓ Numeric types checked"
)

if unpopulated:

    print(
        f"\n⚠ {len(unpopulated)} features "
        "still require real values."
    )

    print(
        "\nNEXT:"
    )

    print(
        "RESOLVE REMAINING FEATURES → "
        "THEN RUN MODEL PREDICTION"
    )

else:

    print(
        "\n✓ ALL 56 FEATURES ARE POPULATED."
    )

    print(
        "\nNEXT:"
    )

    print(
        "STAGE 7 — FINAL MODEL PREDICTION"
    )

print("=" * 70)

STAGE 6 — MULTIMODAL 56-FEATURE ASSEMBLY

LOADING PRODUCTION COMPONENTS

Model:
Pipeline
Expected features: 56
Numeric: 51
Categorical: 5

LOADING STAGE 3 — TEXT FEATURES

Stage 3 shape: (1, 56)

LOADING STAGE 4 — ACCOUNT / POST FEATURES

Stage 4 shape: (1, 56)

LOADING STAGE 5B — IMAGE FEATURES

Stage 5B shape: (1, 56)

INDIVIDUAL FEATURE OUTPUT VALIDATION

Stage 3:
  Rows: 1
  Columns: 56
  Missing: 0
  Unexpected: 0

Stage 4:
  Rows: 1
  Columns: 56
  Missing: 0
  Unexpected: 0

Stage 5B:
  Rows: 1
  Columns: 56
  Missing: 0
  Unexpected: 0

MERGING MULTIMODAL FEATURES
  Existing: 0.0
  New (Stage 4): Travel


TypeError: Invalid value 'Travel' for dtype 'float64'

In [19]:
# ================================================================
# STAGE 6 — CORRECTED MULTIMODAL 56-FEATURE ASSEMBLY
# ================================================================

from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np


print("=" * 70)
print("STAGE 6 — MULTIMODAL 56-FEATURE ASSEMBLY")
print("=" * 70)


# ================================================================
# 1. PROJECT PATH
# ================================================================

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
PRODUCTION_DIR = RESULTS_DIR / "production"

PRODUCTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nProject root:")
print(PROJECT_ROOT)


# ================================================================
# 2. REQUIRED PRODUCTION FILES
# ================================================================

MODEL_PATH = (
    MODELS_DIR /
    "final_instagram_engagement_model.joblib"
)

SCHEMA_PATH = (
    MODELS_DIR /
    "final_model_features.json"
)

METADATA_PATH = (
    MODELS_DIR /
    "final_model_metadata.json"
)


# ================================================================
# 3. VALIDATE MODEL FILES
# ================================================================

print("\n" + "=" * 70)
print("PRODUCTION FILE VALIDATION")
print("=" * 70)

for path in [
    MODEL_PATH,
    SCHEMA_PATH,
    METADATA_PATH
]:

    print(
        f"\n{path.name}:",
        "FOUND" if path.exists() else "NOT FOUND"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Required production file missing:\n{path}"
        )


# ================================================================
# 4. LOAD MODEL + SCHEMA
# ================================================================

print("\n" + "=" * 70)
print("LOADING PRODUCTION MODEL")
print("=" * 70)

model = joblib.load(MODEL_PATH)

with open(
    SCHEMA_PATH,
    "r",
    encoding="utf-8"
) as f:

    schema = json.load(f)

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:

    metadata = json.load(f)


ALL_FEATURES = schema["all_features"]

NUMERIC_FEATURES = schema.get(
    "numeric_features",
    []
)

CATEGORICAL_FEATURES = schema.get(
    "categorical_features",
    []
)


print("\nModel:")
print(type(model).__name__)

print(
    "Expected feature count:",
    len(ALL_FEATURES)
)

print(
    "Numeric features:",
    len(NUMERIC_FEATURES)
)

print(
    "Categorical features:",
    len(CATEGORICAL_FEATURES)
)


if len(ALL_FEATURES) != 56:
    raise ValueError(
        f"Expected 56 features but schema contains "
        f"{len(ALL_FEATURES)}."
    )


# ================================================================
# 5. DISCOVER STAGE OUTPUT FILES
# ================================================================

print("\n" + "=" * 70)
print("DISCOVERING STAGE OUTPUT FILES")
print("=" * 70)


def find_file(filename):
    """
    Search the production results directory first,
    then the entire results directory.
    """

    direct = PRODUCTION_DIR / filename

    if direct.exists():
        return direct

    matches = list(
        RESULTS_DIR.rglob(filename)
    )

    if matches:
        return matches[0]

    return None


stage3_path = find_file(
    "stage3_caption_hashtag_features.csv"
)

stage4_path = find_file(
    "stage4_account_post_features.csv"
)

stage5_path = find_file(
    "stage5b_image_feature_vector.csv"
)


print("\nStage 3:")
print(stage3_path)

print("\nStage 4:")
print(stage4_path)

print("\nStage 5B:")
print(stage5_path)


# ================================================================
# 6. IF STAGE 5B NAME IS DIFFERENT, SEARCH BY KEYWORDS
# ================================================================

if stage5_path is None:

    possible_image_files = []

    for path in RESULTS_DIR.rglob("*"):

        if not path.is_file():
            continue

        name = path.name.lower()

        if (
            "image" in name
            and path.suffix.lower() == ".csv"
        ):

            possible_image_files.append(path)


    print(
        "\nPossible image feature files:"
    )

    for path in possible_image_files:

        print(
            " -",
            path
        )


    if len(possible_image_files) == 1:

        stage5_path = possible_image_files[0]

        print(
            "\nUsing discovered image feature file:"
        )

        print(stage5_path)


# ================================================================
# 7. VALIDATE STAGE FILES
# ================================================================

for stage_name, path in [
    ("Stage 3", stage3_path),
    ("Stage 4", stage4_path),
    ("Stage 5B", stage5_path)
]:

    if path is None:

        raise FileNotFoundError(
            f"{stage_name} output file could not be found."
        )


# ================================================================
# 8. LOAD STAGE DATA
# ================================================================

print("\n" + "=" * 70)
print("LOADING FEATURE SOURCES")
print("=" * 70)


stage3 = pd.read_csv(stage3_path)
stage4 = pd.read_csv(stage4_path)
stage5 = pd.read_csv(stage5_path)


print(
    "\nStage 3 shape:",
    stage3.shape
)

print(
    "Stage 4 shape:",
    stage4.shape
)

print(
    "Stage 5B shape:",
    stage5.shape
)


if len(stage3) == 0:
    raise ValueError("Stage 3 contains no rows.")

if len(stage4) == 0:
    raise ValueError("Stage 4 contains no rows.")

if len(stage5) == 0:
    raise ValueError("Stage 5B contains no rows.")


# ================================================================
# 9. USE FIRST PRODUCTION USER ROW
# ================================================================

stage3_row = stage3.iloc[0]
stage4_row = stage4.iloc[0]
stage5_row = stage5.iloc[0]


# ================================================================
# 10. DEFINE FEATURE OWNERSHIP
# ================================================================

print("\n" + "=" * 70)
print("DEFINING FEATURE OWNERSHIP")
print("=" * 70)


# Text / caption / hashtag features
TEXT_FEATURES = {
    "caption_length",
    "word_count",
    "character_count",
    "hashtag_count",
    "emoji_count",
    "sentiment_score",
    "subjectivity_score",
    "uppercase_count",
    "exclamation_count",
    "question_count",
    "mention_count",
    "url_count",
    "digit_count",
    "special_character_count",
    "average_word_length",
    "unique_word_count",
    "hashtag_density",
    "emoji_density",
}


# Image-related features
IMAGE_KEYWORDS = {
    "image",
    "brightness",
    "contrast",
    "sharpness",
    "saturation",
    "color",
    "edge",
    "texture",
    "aspect_ratio",
    "width",
    "height",
}


def is_image_feature(feature):

    name = feature.lower()

    return any(
        keyword in name
        for keyword in IMAGE_KEYWORDS
    )


IMAGE_FEATURES = {
    feature
    for feature in ALL_FEATURES
    if is_image_feature(feature)
}


# Always explicitly include known image schema fields
for feature in [
    "has_image",
    "image_width",
    "image_height",
    "aspect_ratio",
    "image_brightness",
    "image_contrast",
    "image_sharpness",
]:

    if feature in ALL_FEATURES:

        IMAGE_FEATURES.add(feature)


TEXT_FEATURES = {
    feature
    for feature in TEXT_FEATURES
    if feature in ALL_FEATURES
}


ACCOUNT_POST_FEATURES = {
    feature
    for feature in ALL_FEATURES
    if (
        feature not in TEXT_FEATURES
        and feature not in IMAGE_FEATURES
    )
}


print(
    "\nText features:",
    len(TEXT_FEATURES)
)

for feature in sorted(TEXT_FEATURES):

    print(
        " TEXT:",
        feature
    )


print(
    "\nImage features:",
    len(IMAGE_FEATURES)
)

for feature in sorted(IMAGE_FEATURES):

    print(
        " IMAGE:",
        feature
    )


print(
    "\nAccount/Post features:",
    len(ACCOUNT_POST_FEATURES)
)


# ================================================================
# 11. CREATE MASTER DATAFRAME WITH CORRECT DTYPES
# ================================================================

print("\n" + "=" * 70)
print("CREATING TYPE-SAFE MASTER VECTOR")
print("=" * 70)


master_data = {}

for feature in ALL_FEATURES:

    if feature in CATEGORICAL_FEATURES:

        # Categorical features MUST be object/string.
        master_data[feature] = pd.Series(
            [None],
            dtype="object"
        )

    else:

        # Numeric features MUST be numeric.
        master_data[feature] = pd.Series(
            [np.nan],
            dtype="float64"
        )


master = pd.DataFrame(
    master_data
)


# ================================================================
# 12. HELPER FUNCTION
# ================================================================

def set_feature(
    feature,
    value,
    source_name
):

    if feature not in ALL_FEATURES:
        return

    if pd.isna(value):
        return

    # ------------------------------------------------------------
    # CATEGORICAL FEATURE
    # ------------------------------------------------------------

    if feature in CATEGORICAL_FEATURES:

        value = str(value).strip()

        if value == "":
            return

        current = master.at[0, feature]

        if (
            current is not None
            and not pd.isna(current)
        ):

            if str(current) != value:

                print(
                    f"WARNING: categorical conflict: "
                    f"{feature}"
                )

                print(
                    f" Existing: {current}"
                )

                print(
                    f" New ({source_name}): {value}"
                )

                # Prefer the production account/post
                # value where applicable.
                if source_name == "Stage 4":

                    master.at[
                        0,
                        feature
                    ] = value

        else:

            master.at[
                0,
                feature
            ] = value

        return


    # ------------------------------------------------------------
    # NUMERIC FEATURE
    # ------------------------------------------------------------

    try:

        numeric_value = float(value)

    except Exception:

        raise TypeError(
            f"Feature '{feature}' is numeric but "
            f"received invalid value '{value}' "
            f"from {source_name}."
        )


    current = master.at[0, feature]

    if pd.isna(current):

        master.at[
            0,
            feature
        ] = numeric_value

    else:

        if not np.isclose(
            float(current),
            numeric_value,
            equal_nan=True
        ):

            print(
                f"WARNING: numeric conflict: "
                f"{feature}"
            )

            print(
                f" Existing: {current}"
            )

            print(
                f" New ({source_name}): "
                f"{numeric_value}"
            )

            # Stage 5B has highest priority for
            # image features.
            if (
                source_name == "Stage 5B"
                and feature in IMAGE_FEATURES
            ):

                master.at[
                    0,
                    feature
                ] = numeric_value


# ================================================================
# 13. LOAD ACCOUNT / POST FEATURES
# ================================================================

print("\n" + "=" * 70)
print("ASSEMBLING ACCOUNT / POST FEATURES")
print("=" * 70)


for feature in ACCOUNT_POST_FEATURES:

    if feature in stage4.columns:

        set_feature(
            feature,
            stage4_row[feature],
            "Stage 4"
        )


# ================================================================
# 14. LOAD TEXT FEATURES
# ================================================================

print("\n" + "=" * 70)
print("ASSEMBLING CAPTION / HASHTAG FEATURES")
print("=" * 70)


for feature in TEXT_FEATURES:

    if feature in stage3.columns:

        set_feature(
            feature,
            stage3_row[feature],
            "Stage 3"
        )


# ================================================================
# 15. LOAD IMAGE FEATURES
# ================================================================

print("\n" + "=" * 70)
print("ASSEMBLING IMAGE FEATURES")
print("=" * 70)


for feature in IMAGE_FEATURES:

    if feature in stage5.columns:

        set_feature(
            feature,
            stage5_row[feature],
            "Stage 5B"
        )


# ================================================================
# 16. CHECK FEATURES THAT WERE NOT FOUND IN SOURCES
# ================================================================

print("\n" + "=" * 70)
print("SOURCE COVERAGE VALIDATION")
print("=" * 70)


missing_from_sources = []

for feature in ALL_FEATURES:

    value = master.at[0, feature]

    if pd.isna(value):

        missing_from_sources.append(
            feature
        )


print(
    "\nTotal production features:",
    len(ALL_FEATURES)
)

print(
    "Successfully populated:",
    len(ALL_FEATURES)
    - len(missing_from_sources)
)

print(
    "Still missing:",
    len(missing_from_sources)
)


if missing_from_sources:

    print(
        "\nMissing features:"
    )

    for feature in missing_from_sources:

        feature_type = (
            "CATEGORICAL"
            if feature in CATEGORICAL_FEATURES
            else "NUMERIC"
        )

        print(
            f" - {feature} [{feature_type}]"
        )


# ================================================================
# 17. VALIDATE FEATURE ORDER
# ================================================================

print("\n" + "=" * 70)
print("SCHEMA VALIDATION")
print("=" * 70)


if list(master.columns) != list(
    ALL_FEATURES
):

    raise ValueError(
        "Feature order does not match "
        "final production schema."
    )


print(
    "\n✓ Feature order matches schema."
)


# ================================================================
# 18. VALIDATE FEATURE COUNT
# ================================================================

print(
    "\nExpected features:",
    len(ALL_FEATURES)
)

print(
    "Actual features:",
    len(master.columns)
)


if len(master.columns) != 56:

    raise ValueError(
        "Final production vector does not "
        "contain exactly 56 features."
    )


print(
    "✓ Exactly 56 features present."
)


# ================================================================
# 19. NUMERIC VALIDATION
# ================================================================

print("\n" + "=" * 70)
print("NUMERIC FEATURE VALIDATION")
print("=" * 70)


numeric_errors = []

for feature in NUMERIC_FEATURES:

    value = master.at[0, feature]

    if pd.isna(value):
        continue

    if not isinstance(
        value,
        (int, float, np.integer, np.floating)
    ):

        numeric_errors.append(
            (
                feature,
                value
            )
        )


print(
    "\nNumeric errors:",
    len(numeric_errors)
)


for feature, value in numeric_errors:

    print(
        f" - {feature}: {value}"
    )


if numeric_errors:

    raise TypeError(
        "Numeric feature validation failed."
    )


# ================================================================
# 20. CATEGORICAL VALIDATION
# ================================================================

print("\n" + "=" * 70)
print("CATEGORICAL FEATURE VALIDATION")
print("=" * 70)


categorical_errors = []

for feature in CATEGORICAL_FEATURES:

    value = master.at[0, feature]

    if pd.isna(value):
        continue

    if not isinstance(
        value,
        str
    ):

        categorical_errors.append(
            (
                feature,
                value,
                type(value).__name__
            )
        )


print(
    "\nCategorical errors:",
    len(categorical_errors)
)


for feature, value, dtype in categorical_errors:

    print(
        f" - {feature}: {value} "
        f"({dtype})"
    )


if categorical_errors:

    raise TypeError(
        "Categorical feature validation failed."
    )


# ================================================================
# 21. DUPLICATE FEATURE CHECK
# ================================================================

print("\n" + "=" * 70)
print("DUPLICATE FEATURE CHECK")
print("=" * 70)


duplicates = master.columns[
    master.columns.duplicated()
].tolist()


print(
    "\nDuplicate features:",
    len(duplicates)
)


if duplicates:

    print(duplicates)

    raise ValueError(
        "Duplicate production features detected."
    )


print(
    "✓ No duplicate features."
)


# ================================================================
# 22. DISPLAY FINAL VECTOR
# ================================================================

print("\n" + "=" * 70)
print("FINAL MULTIMODAL 56-FEATURE VECTOR")
print("=" * 70)


for index, feature in enumerate(
    ALL_FEATURES,
    start=1
):

    value = master.at[
        0,
        feature
    ]

    print(
        f"{index:02d}. "
        f"{feature:<42} = {value}"
    )


# ================================================================
# 23. SAVE ASSEMBLED VECTOR
# ================================================================

OUTPUT_FILE = (
    PRODUCTION_DIR /
    "stage6_multimodal_56_feature_vector.csv"
)


master.to_csv(
    OUTPUT_FILE,
    index=False
)


print("\n" + "=" * 70)
print("STAGE 6 OUTPUT SAVED")
print("=" * 70)

print(
    OUTPUT_FILE
)


# ================================================================
# 24. FINAL STATUS
# ================================================================

print("\n" + "=" * 70)
print("STAGE 6 — FINAL STATUS")
print("=" * 70)


if missing_from_sources:

    print(
        "\n⚠ STAGE 6 ASSEMBLY INCOMPLETE"
    )

    print(
        f"{len(missing_from_sources)} "
        "features still require values."
    )

    print(
        "\nDO NOT RUN THE MODEL YET."
    )

    print(
        "We will resolve these features first."
    )

else:

    print(
        "\n✓ ALL 56 FEATURES POPULATED"
    )

    print(
        "✓ Text features integrated"
    )

    print(
        "✓ Account/post features integrated"
    )

    print(
        "✓ Image features integrated"
    )

    print(
        "✓ Numeric types validated"
    )

    print(
        "✓ Categorical types validated"
    )

    print(
        "✓ Feature order validated"
    )

    print(
        "✓ Production vector ready"
    )

    print(
        "\nNEXT: STAGE 7 — FINAL MODEL PREDICTION"
    )

print("=" * 70)

STAGE 6 — MULTIMODAL 56-FEATURE ASSEMBLY

Project root:
d:\newwwwwwww\AiBasedInstagramPrediction

PRODUCTION FILE VALIDATION

final_instagram_engagement_model.joblib: FOUND

final_model_features.json: FOUND

final_model_metadata.json: FOUND

LOADING PRODUCTION MODEL

Model:
Pipeline
Expected feature count: 56
Numeric features: 51
Categorical features: 5

DISCOVERING STAGE OUTPUT FILES

Stage 3:
d:\newwwwwwww\AiBasedInstagramPrediction\results\production\stage3_caption_hashtag_features.csv

Stage 4:
d:\newwwwwwww\AiBasedInstagramPrediction\results\production\stage4_account_post_features.csv

Stage 5B:
d:\newwwwwwww\AiBasedInstagramPrediction\results\production\stage5b_image_feature_vector.csv

LOADING FEATURE SOURCES

Stage 3 shape: (1, 56)
Stage 4 shape: (1, 56)
Stage 5B shape: (1, 56)

DEFINING FEATURE OWNERSHIP

Text features: 6
 TEXT: caption_length
 TEXT: emoji_count
 TEXT: exclamation_count
 TEXT: hashtag_count
 TEXT: mention_count
 TEXT: word_count

Image features: 11
 IMAGE: asp

In [20]:
# ================================================================
# STAGE 7 — FINAL MULTIMODAL MODEL PREDICTION
# ================================================================

from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np


print("=" * 70)
print("STAGE 7 — FINAL MULTIMODAL MODEL PREDICTION")
print("=" * 70)


# ================================================================
# 1. PROJECT PATH
# ================================================================

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

MODELS_DIR = PROJECT_ROOT / "models"
PRODUCTION_DIR = PROJECT_ROOT / "results" / "production"

PRODUCTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 2. FILES
# ================================================================

MODEL_PATH = (
    MODELS_DIR /
    "final_instagram_engagement_model.joblib"
)

SCHEMA_PATH = (
    MODELS_DIR /
    "final_model_features.json"
)

METADATA_PATH = (
    MODELS_DIR /
    "final_model_metadata.json"
)

VECTOR_PATH = (
    PRODUCTION_DIR /
    "stage6_multimodal_56_feature_vector.csv"
)


# ================================================================
# 3. VALIDATION
# ================================================================

print("\n" + "=" * 70)
print("FILE VALIDATION")
print("=" * 70)

required_files = {
    "Model": MODEL_PATH,
    "Feature schema": SCHEMA_PATH,
    "Metadata": METADATA_PATH,
    "Stage 6 vector": VECTOR_PATH
}

for name, path in required_files.items():

    exists = path.exists()

    print(
        f"\n{name}:",
        "FOUND" if exists else "NOT FOUND"
    )

    print(path)

    if not exists:

        raise FileNotFoundError(
            f"{name} is missing:\n{path}"
        )


# ================================================================
# 4. LOAD COMPONENTS
# ================================================================

print("\n" + "=" * 70)
print("LOADING FINAL PRODUCTION MODEL")
print("=" * 70)

model = joblib.load(
    MODEL_PATH
)

with open(
    SCHEMA_PATH,
    "r",
    encoding="utf-8"
) as f:

    schema = json.load(f)

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:

    metadata = json.load(f)


ALL_FEATURES = schema["all_features"]

NUMERIC_FEATURES = schema.get(
    "numeric_features",
    []
)

CATEGORICAL_FEATURES = schema.get(
    "categorical_features",
    []
)


print(
    "\nModel type:",
    type(model).__name__
)

print(
    "Expected features:",
    len(ALL_FEATURES)
)

print(
    "Numeric features:",
    len(NUMERIC_FEATURES)
)

print(
    "Categorical features:",
    len(CATEGORICAL_FEATURES)
)


# ================================================================
# 5. LOAD STAGE 6 VECTOR
# ================================================================

print("\n" + "=" * 70)
print("LOADING STAGE 6 MULTIMODAL VECTOR")
print("=" * 70)

X = pd.read_csv(
    VECTOR_PATH
)

print(
    "\nVector shape:",
    X.shape
)


# ================================================================
# 6. FEATURE COUNT CHECK
# ================================================================

print("\n" + "=" * 70)
print("FEATURE SCHEMA CHECK")
print("=" * 70)

print(
    "\nExpected:",
    len(ALL_FEATURES)
)

print(
    "Received:",
    len(X.columns)
)


if len(X.columns) != 56:

    raise ValueError(
        f"Expected 56 features but received "
        f"{len(X.columns)}."
    )


# ================================================================
# 7. FEATURE ORDER CHECK
# ================================================================

if list(X.columns) != list(ALL_FEATURES):

    missing = [
        f
        for f in ALL_FEATURES
        if f not in X.columns
    ]

    unexpected = [
        f
        for f in X.columns
        if f not in ALL_FEATURES
    ]

    print(
        "\nMissing:",
        missing
    )

    print(
        "Unexpected:",
        unexpected
    )

    raise ValueError(
        "Feature order/schema does not match "
        "the production model."
    )


print(
    "\n✓ Feature order matches production schema."
)


# ================================================================
# 8. MISSING VALUE CHECK
# ================================================================

print("\n" + "=" * 70)
print("MISSING VALUE CHECK")
print("=" * 70)

missing_values = X.isna().sum()

missing_features = (
    missing_values[
        missing_values > 0
    ]
)


if len(missing_features) > 0:

    print(
        "\nMissing values detected:"
    )

    print(
        missing_features
    )

    raise ValueError(
        "Stage 6 vector contains missing values. "
        "Prediction stopped."
    )


print(
    "\n✓ No missing values."
)


# ================================================================
# 9. INFINITE VALUE CHECK
# ================================================================

print("\n" + "=" * 70)
print("NUMERIC VALUE VALIDATION")
print("=" * 70)

numeric_X = X[
    NUMERIC_FEATURES
].apply(
    pd.to_numeric,
    errors="coerce"
)


if numeric_X.isna().any().any():

    invalid = []

    for feature in NUMERIC_FEATURES:

        if numeric_X[feature].isna().any():

            invalid.append(feature)

    raise TypeError(
        "Invalid numeric values found:\n"
        + "\n".join(invalid)
    )


if np.isinf(
    numeric_X.to_numpy()
).any():

    raise ValueError(
        "Infinite numeric value detected."
    )


print(
    "✓ Numeric values valid."
)


# ================================================================
# 10. CATEGORICAL VALUE CHECK
# ================================================================

print("\n" + "=" * 70)
print("CATEGORICAL VALUE VALIDATION")
print("=" * 70)

for feature in CATEGORICAL_FEATURES:

    value = X.iloc[0][feature]

    print(
        f"{feature}: {value}"
    )


print(
    "\n✓ Categorical values loaded."
)


# ================================================================
# 11. MODEL INPUT
# ================================================================

print("\n" + "=" * 70)
print("PREPARING MODEL INPUT")
print("=" * 70)


# Preserve exact production feature order.
X_model = X[
    ALL_FEATURES
].copy()


print(
    "\nModel input shape:",
    X_model.shape
)


# ================================================================
# 12. FINAL MODEL PREDICTION
# ================================================================

print("\n" + "=" * 70)
print("RUNNING FINAL MODEL")
print("=" * 70)


prediction = model.predict(
    X_model
)


predicted_class = str(
    prediction[0]
)


print(
    "\nPredicted engagement:",
    predicted_class
)


# ================================================================
# 13. PREDICTION PROBABILITIES
# ================================================================

probabilities = None

if hasattr(
    model,
    "predict_proba"
):

    probabilities = model.predict_proba(
        X_model
    )[0]


elif hasattr(
    model,
    "named_steps"
):

    # For sklearn Pipeline
    if hasattr(
        model,
        "predict_proba"
    ):

        probabilities = model.predict_proba(
            X_model
        )[0]


# ================================================================
# 14. DISPLAY PROBABILITIES
# ================================================================

print("\n" + "=" * 70)
print("ENGAGEMENT PROBABILITIES")
print("=" * 70)


probability_dict = {}


if probabilities is not None:

    classes = getattr(
        model,
        "classes_",
        None
    )


    # Pipeline models may expose classes
    # through the final estimator.
    if classes is None and hasattr(
        model,
        "named_steps"
    ):

        final_estimator = list(
            model.named_steps.values()
        )[-1]

        classes = getattr(
            final_estimator,
            "classes_",
            None
        )


    if classes is None:

        classes = [
            "High",
            "Low",
            "Medium"
        ]


    for cls, probability in zip(
        classes,
        probabilities
    ):

        probability = float(
            probability
        )

        probability_dict[
            str(cls)
        ] = probability


        print(
            f"{str(cls):<10} : "
            f"{probability:.4f} "
            f"({probability * 100:.2f}%)"
        )


else:

    print(
        "Probability output is not available "
        "for this model."
    )


# ================================================================
# 15. CONFIDENCE
# ================================================================

if probability_dict:

    confidence = max(
        probability_dict.values()
    )

else:

    confidence = None


print(
    "\nPredicted class:",
    predicted_class
)

if confidence is not None:

    print(
        "Confidence:",
        f"{confidence * 100:.2f}%"
    )


# ================================================================
# 16. CREATE PRODUCTION RESULT
# ================================================================

result = {
    "project_name": metadata.get(
        "project_name",
        "AI-Based Instagram Engagement Prediction and Content Optimization System"
    ),

    "model_name": type(model).__name__,

    "prediction": predicted_class,

    "confidence": (
        round(confidence, 6)
        if confidence is not None
        else None
    ),

    "probabilities": probability_dict,

    "feature_count": len(ALL_FEATURES),

    "input_type": "multimodal",

    "status": "success"
}


# ================================================================
# 17. SAVE JSON RESULT
# ================================================================

RESULT_JSON = (
    PRODUCTION_DIR /
    "stage7_final_prediction.json"
)

with open(
    RESULT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result,
        f,
        indent=4
    )


# ================================================================
# 18. SAVE CSV RESULT
# ================================================================

RESULT_CSV = (
    PRODUCTION_DIR /
    "stage7_final_prediction.csv"
)

result_row = {
    "prediction": predicted_class,
    "confidence": confidence,
    "feature_count": len(ALL_FEATURES),
    "input_type": "multimodal"
}

for cls, probability in probability_dict.items():

    result_row[
        f"probability_{cls.lower()}"
    ] = probability


pd.DataFrame(
    [result_row]
).to_csv(
    RESULT_CSV,
    index=False
)


# ================================================================
# 19. FINAL OUTPUT
# ================================================================

print("\n" + "=" * 70)
print("STAGE 7 — FINAL PREDICTION COMPLETED")
print("=" * 70)

print(
    "\n🎯 PREDICTION:",
    predicted_class
)

if confidence is not None:

    print(
        "🎯 CONFIDENCE:",
        f"{confidence * 100:.2f}%"
    )

print(
    "\nJSON result:"
)

print(
    RESULT_JSON
)

print(
    "\nCSV result:"
)

print(
    RESULT_CSV
)

print("\n" + "=" * 70)

print(
    "NEXT: STAGE 8 — PRODUCTION PREDICTION PIPELINE"
)

print("=" * 70)

STAGE 7 — FINAL MULTIMODAL MODEL PREDICTION

FILE VALIDATION

Model: FOUND
d:\newwwwwwww\AiBasedInstagramPrediction\models\final_instagram_engagement_model.joblib

Feature schema: FOUND
d:\newwwwwwww\AiBasedInstagramPrediction\models\final_model_features.json

Metadata: FOUND
d:\newwwwwwww\AiBasedInstagramPrediction\models\final_model_metadata.json

Stage 6 vector: FOUND
d:\newwwwwwww\AiBasedInstagramPrediction\results\production\stage6_multimodal_56_feature_vector.csv

LOADING FINAL PRODUCTION MODEL

Model type: Pipeline
Expected features: 56
Numeric features: 51
Categorical features: 5

LOADING STAGE 6 MULTIMODAL VECTOR

Vector shape: (1, 56)

FEATURE SCHEMA CHECK

Expected: 56
Received: 56

✓ Feature order matches production schema.

MISSING VALUE CHECK

✓ No missing values.

NUMERIC VALUE VALIDATION
✓ Numeric values valid.

CATEGORICAL VALUE VALIDATION
category: Travel
account_type: Creator
day_of_week: Saturday
posting_time_period: Evening
media_type: Image

✓ Categorical values l